### Packages

In [ ]:
# Imports
# > Private packages
from image_io import *
from calculations import *
from consts import *
from plotting import *
# > Public packages
import anndata as ad
import ast
import h5py
import hdbscan
import igraph as ig
import leidenalg as la
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import phate
import scanpy as sc
import scipy.stats as ss
import seaborn as sns
import torch
import umap
from sklearn.cluster import DBSCAN, KMeans
from sklearn.neighbors import kneighbors_graph
from sklearn.preprocessing import StandardScaler
from scipy.stats import zscore
from tqdm import tqdm

# Adjust Plot Resolution
# > If in IPython notebook utilize higher resolution 'retina' format
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('retina')
# > Adjust the resolution (dots-per-inch, DPI)
from matplotlib import rcParams
rcParams['figure.dpi'] = 100

### Fig. 1B (Representative Image)

In [ ]:
# Read in the segmentation and marker intensities file
# > Define file name
segmentation_file = '/PATH/TO/SEGMENTATION_TIFF'
intensities_file = '/PATH/TO/INTENSITIES_TIFF'
# > Process data in files
segmentation_imstack, marker_imstack = load_data(segmentation_file, intensities_file)
segmentation_dict = create_segmentation_dict(segmentation_imstack)
# > Define marker of interest and locate in image stack
marker_of_interest = 'Vimentin'
marker_list = create_marker_list_mibiv2(intensities_file)
image_stack_index = marker_list.index(marker_of_interest)
# > Retrieve intensities for the marker of interest
intensities = marker_imstack[0][image_stack_index]
# > Set accelerator device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
intensities_acc = torch.tensor(intensities, device=device, dtype=torch.float32)

# Generate smoothed vectors
vectors = generate_vector_field(segmentation_dict, intensities_acc, device)
smoothed_vectors = smooth_vector_field(vectors,
                                       min_magnitude=MIN_VECTOR_MAGNITUDE,
                                       smooth_iters=SMOOTH_ITERS,
                                       smooth_dist_limit_pixel=SMOOTH_DIST_LIMIT_PIXEL)

# Read in the expression and convert to an AnnData object
X = pd.read_table('/PATH/TO/INTENSITIES_TSV')
X = X.pivot_table(index='segment', columns='marker', values='sum_segmentint').fillna(0)
adata = ad.AnnData(X)
# Read in assigned cell types (see Campbell et al. 2026, Cancer Discovery for MIBI data)
cell_types = pd.read_csv('/PATH/TO/CELL_TYPES_CSV', index_col=0)
cell_types.index = cell_types.index.astype(str)
adata.obs['cell_type'] = cell_types
# Add in raw and smoothed vectors along with spatial positions
spatial = vectors.set_index('segment')[['x0','y0','magnitude']]
spatial.index = spatial.index.astype(str)
adata.obs[spatial.columns] = spatial
adata.obsm['spatial'] = adata.obs[['x0','y0']].values
movement = smoothed_vectors.set_index('segment')[['unit_u','unit_v']]
movement.index = movement.index.astype(str)
adata.obs[movement.columns] = movement

# Project polarization vectors onto the data
fig, ax = plt.subplots(figsize=[8, 8]); _ = ax.grid(False)
ax = sc.pl.spatial(adata, spot_size=0, show=False, ax=ax)
ax = sc.pl.spatial(adata, spot_size=20, ax=ax[0], color=['cell_type'], show=False, alpha=0.8)[0]
ax.quiver(adata.obs['x0'], adata.obs['y0'], adata.obs['unit_u'], adata.obs['unit_v'],
          width=QUIVER_WIDTH * 0.8, scale=QUIVER_SCALE / 0.8, alpha=0.8)

### Fig. S1 (Actin Polarization)

In [ ]:
from glob import glob
from scipy.ndimage import label
from tqdm import tqdm
# Retrieve all segmentation files (data from Weiner et al. 2010, Nature Cell Biology)
segmentation_files = sorted(glob('/PATH/TO/MOVIE_TIFFS'))
# Instantiate a tracker for all vectors
vectors = []
# Read in the segmentation and marker intensities file
prev_centroid = np.zeros(shape = 2)
for segmentation_file in tqdm(segmentation_files):
    # > Define file name
    intensities_file = segmentation_file.replace('segmentations','intensities')
    name = int(segmentation_file.split('/')[-1].split('movie_')[-1].split('.tiff')[0])
    # > Process data in files
    segmentation_imstack, marker_imstack = load_data(segmentation_file, intensities_file)
    # > Assign labels to the mask
    labeled_mask, num_labels = label(segmentation_imstack)
    segmentation_dict = create_segmentation_dict(labeled_mask)
    # > Find the correct cell
    minimum_dist, minimum_id, minimum_centroid = np.inf, -1, None
    for k,v in segmentation_dict.items():
        if len(v) < 5: continue    
        centroid = np.array(v).mean(axis=0)
        dist_from_upperleft = np.sqrt((np.mean(centroid - prev_centroid) ** 2))
        if dist_from_upperleft < minimum_dist:
            minimum_dist = dist_from_upperleft
            minimum_id = k
            minimum_centroid = centroid
    labeled_mask[(labeled_mask > 0) & (labeled_mask != minimum_id)] = 0
    labeled_mask[labeled_mask > 0] = 1
    segmentation_dict = create_segmentation_dict(labeled_mask)
    prev_centroid = minimum_centroid
    # > Retrieve intensities for the marker of interest
    intensities = marker_imstack[0]
    # > Set accelerator device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    intensities_acc = torch.tensor(intensities, device=device, dtype=torch.float32)
    # Generate vector
    vector = generate_vector_field(segmentation_dict, intensities_acc, device)
    vector = vector.iloc[0]
    vector.name = name
    vectors.append(vector)
# Assemble the vectors together
vectors = pd.concat(vectors, axis=1).T.astype(float)

# Highlight the path of the cell over time
fig, ax = plt.subplots(figsize=[4, 4]); ax.grid(False)
# > Indicate the start
ax.scatter(vectors['x0'].iloc[0], vectors['y0'].iloc[0], edgecolor='navy', color='dodgerblue', marker='o', zorder=2)
# > Plot the path
ax.scatter(vectors['x0'], vectors['y0'], edgecolor='dodgerblue', color='skyblue', alpha=0.9, zorder=1)
ax.plot(vectors['x0'], vectors['y0'], color='dodgerblue', lw=1, linestyle='--', zorder=1)
# > Indicate the end
ax.scatter(vectors['x0'].iloc[-1], vectors['y0'].iloc[-1], edgecolor='navy', color='dodgerblue', marker='s', zorder=2)
ax.set(xlabel='Position X', ylabel='Position Y')

In [ ]:
# Calculate the true delta x and y
vectors[['dx','dy']] = np.nan
for idx, xlabel1 in enumerate(vectors.index[:-1]):
    xlabel2 = vectors.index[idx + 1]
    x1, y1 = vectors.loc[xlabel1, ['x0','y0']]
    x2, y2 = vectors.loc[xlabel2, ['x0','y0']]
    dx, dy = (x2 - x1), (y2 - y1)
    vectors.loc[xlabel1, ['dx','dy']] = dx, dy

# Compute the angles across time
vectors['dangle'] = (np.arctan(vectors['dy'] / vectors['dx']) + np.pi) % np.pi
vectors['vangle'] = (np.arctan(vectors['yv'] / vectors['xv']) + np.pi) % np.pi
vectors.loc[vectors['dx'] < 0, 'dangle'] += np.pi
vectors['dangle'] = (vectors['dangle'] + 2 * np.pi) % (2 * np.pi)
vectors['vangle'] = np.arctan(vectors['yv'] / vectors['xv'])
vectors.loc[vectors['xv'] < 0, 'vangle'] += np.pi
vectors['vangle'] = (vectors['vangle'] + 2 * np.pi) % (2 * np.pi)

# Project movement across time
fig, ax = plt.subplots(figsize=[6, 3]); ax.grid(False)
ax.plot(vectors['dangle'], color='dodgerblue', alpha=0.9, label='True')
ax.plot(vectors['vangle'], color='skyblue', alpha=0.9, linestyle='--', label='Pred.')
ax.set(xlabel='Time', ylabel='Angle')

In [ ]:
# Compare predictions versus ground truth
fig, ax = plt.subplots(figsize=[2, 4]); ax.grid(False)
sns.boxplot(x=['True'] * (vectors.shape[0] - 1) + ['Pred'] * (vectors.shape[0] - 1),
            y=vectors['dangle'].iloc[:-1].tolist() + vectors['vangle'].iloc[:-1].tolist(),
            linecolor='dodgerblue', color='skyblue', saturation=1, linewidth=1.5, showfliers=False)
sns.stripplot(x=['True'] * (vectors.shape[0] - 1) + ['Pred'] * (vectors.shape[0] - 1),
              y=vectors['dangle'].iloc[:-1].tolist() + vectors['vangle'].iloc[:-1].tolist(),
              edgecolor='dodgerblue', color='skyblue', linewidth=1.5, ax=ax, jitter=0.3, s=2, alpha=0.5)
ax.set_xlim(-0.8, 1.8)
ax.set(ylabel='Angle')

### Fig. 2 (Technical Validation)

In [ ]:
from glob import glob
from tqdm import tqdm
# Gather all files from Campbell et al. 2026, Cancer Discovery
fns = glob('/PATH/TO/TIFF_FILES')
fn2vector = {}
for fn in tqdm(fns):
    # Add in special naming
    dirname1 = fn.split('/')[-2]
    dirname2 = dirname1[:6] + '_' + dirname1[7:]
    # Read in the segmentation and marker intensities file
    # > Define file name
    segmentation_file = f'/PATH/TO/TIFF_FILES/{dirname1}/{dirname2}_expanded_full_mask.tif'
    intensities_file = f'/PATH/TO/TIFF_FILES/{dirname1}/{dirname1}.tiff'
    # > Process data in files
    segmentation_imstack, marker_imstack = load_data(segmentation_file, intensities_file)
    segmentation_dict = create_segmentation_dict(segmentation_imstack)
    # > Define marker of interest and locate in image stack
    marker_of_interest = 'Vimentin'
    marker_list = create_marker_list_mibiv2(intensities_file)
    image_stack_index = marker_list.index(marker_of_interest)
    # > Retrieve intensities for the marker of interest
    intensities = marker_imstack[0][image_stack_index]
    # > Set accelerator device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    intensities_acc = torch.tensor(intensities, device=device, dtype=torch.float32)
    
    # Generate smoothed vectors
    vectors = generate_vector_field(segmentation_dict, intensities_acc, device)
    # read in the cell types
    cell_types = pd.read_csv(f'/PATH/TO/CELL_TYPES/{dirname2}_expanded_segment_types.csv', index_col=0)['type']
    cell_types.index = cell_types.index.astype(str)
    vectors['celltype'] = vectors['segment'].astype(str).map(cell_types)
    fn2vector[fn] = vectors
# FN2VECTOR was then written into an intermediate file that is provided in the Zenodo

In [ ]:
from scipy.interpolate import LinearNDInterpolator
# define the conversion factor between pixels
pixels = 90 / 0.390625
# create a tracking object for the filename to metric mapping
fn2data = {}
# loop through each melanoma MIBI object
for fn in tqdm(fn2vector):
    # occasionally there are vectors that are too sparse / few cells to run the curl and divergence
    # calculations and are therefore skipped with this large try and except phrase
    try:
        # read in the vector object
        vector = fn2vector[fn]
        # smooth the vectors (90 microns)
        smooth = smooth_vector_field(vector, min_magnitude=1, smooth_iters=1, smooth_dist_limit_pixel=pixels)
        # instantiate tracking objects
        xs, ys = None, None
        curls, divs = [], []
        # loop through a range of grid sizes
        segs = [el for el in range(100, 10-1, -10)]
        for ngridx in segs:
            # calculate curl and divergence
            ngridy = ngridx
            XI, YI, curl, divergence = calculate_curl_and_divergence(smooth, ngridx, ngridy)
            # gather coordinates
            if ngridx == 100:
                xs_flat, ys_flat = XI.flatten(), YI.flatten()
                xs = np.linspace(xs_flat.min(), xs_flat.max())
                ys = np.linspace(ys_flat.min(), ys_flat.max())
            # train the interpolator and execute it
            lni = LinearNDInterpolator(np.array([XI.flatten(), YI.flatten()]).T, curl.flatten())
            z = np.nan_to_num(lni(*np.meshgrid(xs, ys)).flatten(), 0)
            z -= z.mean(); z /= z.std()
            curls.append(pd.Series(z, name=ngridx))
            # repeat for divergence
            lni = LinearNDInterpolator(np.array([XI.flatten(), YI.flatten()]).T, divergence.flatten())
            z = np.nan_to_num(lni(*np.meshgrid(xs, ys)).flatten(), 0)
            z -= z.mean(); z /= z.std()
            divs.append(pd.Series(z, name=ngridx))
        # highlight absolute difference in z-score
        c2c, d2d = pd.DataFrame(), pd.DataFrame()
        for idx1, seg1 in enumerate(segs):
            for idx2, seg2 in enumerate(segs):
                c2c.loc[seg1, seg2] = (curls[idx1].fillna(0) - curls[idx2].fillna(0)).abs().mean()
                d2d.loc[seg1, seg2] = (divs[idx1].fillna(0) - divs[idx2].fillna(0)).abs().mean()
        # save calculations to the tracking object
        fn2data[fn] = c2c, d2d
    except:
        pass

# display the average values
avg_c2c = pd.concat([fn2data[el][0] for el in fn2data], axis=0).reset_index().groupby('index').mean()
avg_d2d = pd.concat([fn2data[el][1] for el in fn2data], axis=0).reset_index().groupby('index').mean()
fig, ax = plt.subplots(figsize=[4.8, 4])
_ = ax.grid(False)
_ = sns.heatmap(avg_c2c, ax=ax, cmap='Blues', cbar_kws={'label': 'Absolute Difference in Z-Score'}, vmin=0, vmax=2)
_ = ax.set(xlabel='Grid Size for Curl', ylabel='Grid Size for Curl')
fig, ax = plt.subplots(figsize=[4.8, 4])
_ = ax.grid(False)
_ = sns.heatmap(avg_d2d, ax=ax, cmap='Blues', cbar_kws={'label': 'Absolute Difference in Z-Score'}, vmin=0, vmax=2)
_ = ax.set(xlabel='Grid Size for Divergence', ylabel='Grid Size for Divergence')

### Fig. 2 and S2 (Mouse Pup Validation)

In [ ]:
from collections import defaultdict
from tqdm import tqdm
# Read in cell boundary data (from 10x Genomics)
df = pd.read_csv('/PATH/TO/XENIUM_MOUSE_PUP/CELL_BOUNDARIES_CSV')
# Map cell id to segmentation id
cell_ids = df['cell_id'].unique()
c2i = {x: i for i, x in enumerate(cell_ids)}
df['cell_id'] = df['cell_id'].map(c2i)
df = df[['vertex_x','vertex_y','cell_id']].values
# Convert micron to pixels
df[:, :2] /= 0.2125
# Instantiate tracking dictionary
segmentation_dict = defaultdict(list)
# Process all cell ids
prev_cell_id = None
for idx in tqdm(range(df.shape[0])):
    x, y, cell_id = df[idx, :]
    segmentation_dict[int(cell_id)].append([x, y])

# Path to the pyramid OME-TIFF
marker_file = '/PATH/TO/XENIUM_MOUSE_PUP/MORPHOLOGY_TIFF'
# Derive list of marker genes from the XML
print(f'Reading in TIF files from {marker_file}')
with tifffile.TiffFile(marker_file) as tif:
    ome_metadata = tif.ome_metadata
root = ET.fromstring(ome_metadata)
namespace = {'ome': 'http://www.openmicroscopy.org/Schemas/OME/2016-06'}
# Find all channel names
channel_names = []
for image in root.findall('ome:Image', namespace):
    for channel in image.find('ome:Pixels', namespace).findall('ome:Channel', namespace):
        name = channel.get('Name')
        channel_names.append(name)
# Retrieve full image
print(f'Loading OME-TIFF stack from: {marker_file}')
with tifffile.TiffFile(marker_file) as tif:
    marker_imstack = tif.asarray()
    num_layers, img_height, img_width = marker_imstack.shape
    print(f'Loaded image stack with {num_layers} layers, height: {img_height}, width: {img_width}')

# Retrieve intensities (Y, X)
intensities = marker_imstack[channel_names.index('alphaSMA/Vimentin'), :, :]
del marker_imstack
# Set accelerator device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
intensities_acc = torch.tensor(intensities, device=device, dtype=torch.float32)
# Generate vector
vectors = generate_vector_field(segmentation_dict, intensities_acc, device)

# define chunk parameters
chunk_size = 2500
start, end = 0, chunk_size
# instantiate new columns in vectors
keys = ['smooth_u','smooth_v','smooth_magnitude','unit_u','unit_v']
vectors[keys] = np.nan
# process each chunk
while start < vectors.shape[0]:
    # subset for the relevant data
    mask = (vectors.index >= start) & (vectors.index < end)
    df = vectors.loc[mask]
    # calculate distance matrix between each object
    dist_segment = cdist(df[['x0','y0']], df[['x0','y0']])
    dist_segment = pd.DataFrame(dist_segment, index=df.index, columns=df.index)
    # only consider vectors above a minimum magnitude
    mask = df['magnitude'] >= MIN_VECTOR_MAGNITUDE
    df = df.loc[mask]
    # smooth vectors by spatial distance
    df['smooth_u'] = spatial_smooth(df['xv'], SMOOTH_ITERS, dist_segment, SMOOTH_DIST_LIMIT_MICRONS / 0.2125)
    df['smooth_v'] = spatial_smooth(df['yv'], SMOOTH_ITERS, dist_segment, SMOOTH_DIST_LIMIT_MICRONS / 0.2125)
    # normalize into unit vectors
    df['smooth_magnitude'] = np.linalg.norm(df[['smooth_u', 'smooth_v']], axis=1)
    df['unit_u'] = df['smooth_u'] / df['smooth_magnitude']
    df['unit_v'] = df['smooth_v'] / df['smooth_magnitude']
    # add to the object
    vectors.loc[df.index, keys] = df[keys]
    # proceed to next chunk
    start += chunk_size
    end += chunk_size

In [ ]:
import scanpy as sc
# read in the data
adata = sc.read_10x_h5('/PATH/TO/XENIUM_MOUSE_PUP/CELL_FEATURE_H5')
# add in vectors data
adata.obs['seg_id'] = adata.obs.index.map(c2i)
adata.obs[vectors.columns] = vectors.loc[adata.obs['seg_id']].values
# add in spatial coordinates
adata.obsm['spatial'] = adata.obs[['x0','y0']].values
# save raw counts
adata.layers['raw_counts'] = adata.X.copy()
# normalize data
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
# setup matplotlib figure for the overall mouse pup
fig, ax = plt.subplots(figsize=[np.diff(xlim)[0] / 10000, np.diff(ylim)[0] / 10000])
ax.grid(False)
# randomly sample quivers to avoid overwhelming the system
np.random.seed(0)
idxs = np.random.choice(vectors.index, size=50000, replace=False)
# plot quivers
ax.scatter(vectors['x0'], vectors['y0'], color='lightgray', s=0.01)
ax.quiver(vectors.loc[idxs, 'x0'], vectors.loc[idxs, 'y0'], vectors.loc[idxs, 'unit_u'], vectors.loc[idxs, 'unit_v'],
          width=QUIVER_WIDTH * 0.5, scale=QUIVER_SCALE / 0.5)
xlim, ylim = ax.get_xlim(), ax.get_ylim()

# setup matplotlib figure for the mouse paw
fig, ax = plt.subplots(figsize=[np.diff(xlim)[0] / 500, np.diff(ylim)[0] / 500])
ax.grid(False)
# randomly sample quivers to avoid overwhelming the system
mask = (vectors['x0'] >= 4.6e4) & (vectors['x0'] < 5e4)
mask = mask & (vectors['y0'] >= 6.2e4) & (vectors['y0'] < 6.6e4)
# plot quivers
ax.scatter(vectors.loc[mask, 'x0'], vectors.loc[mask, 'y0'], color='lightgray', s=2)
ax.quiver(vectors.loc[mask, 'x0'], vectors.loc[mask, 'y0'], vectors.loc[mask, 'unit_u'], vectors.loc[mask, 'unit_v'],
          width=QUIVER_WIDTH * 0.75, scale=QUIVER_SCALE / 0.75)
xlim, ylim = ax.get_xlim(), ax.get_ylim()

# setup matplotlib figure for the section of the mouse paw
fig, ax = plt.subplots(figsize=[np.diff(xlim)[0] / 200, np.diff(ylim)[0] / 200])
ax.grid(False)
# randomly sample quivers to avoid overwhelming the system
mask = (vectors['x0'] >= 4.7e4) & (vectors['x0'] < 4.8e4)
mask = mask & (vectors['y0'] >= 6.43e4) & (vectors['y0'] < 6.53e4)
# plot quivers
ax.scatter(vectors.loc[mask, 'x0'], vectors.loc[mask, 'y0'], color='lightgray', s=10)
ax.quiver(vectors.loc[mask, 'x0'], vectors.loc[mask, 'y0'], vectors.loc[mask, 'unit_u'], vectors.loc[mask, 'unit_v'],
          width=QUIVER_WIDTH * 2, scale=QUIVER_SCALE / 2)
xlim, ylim = ax.get_xlim(), ax.get_ylim()

# subsample data and plot the defining markers for the developing paw
mask = (vectors['x0'] >= 4.6e4) & (vectors['x0'] < 5e4)
mask = mask & (vectors['y0'] >= 6.2e4) & (vectors['y0'] < 6.6e4)
# a basal plot is required to flip the axes
for gene in ['Dsc3','Scox9','Mki67','Pcna','Lhx2','Cdh2','Thbs2','Runx2','Epcam','Meis2']:
    if gene not in adata.var.index.tolist(): continue
    ax = sc.pl.spatial(adata[mask], spot_size=0, show=False)
    ax = sc.pl.spatial(adata[mask], spot_size=5e1, ax=ax[0], color=[gene], vmin=-1, vmax=7, cmap='Blues')

# subsample data for the region of interst
mask = (vectors['x0'] >= 4.7e4) & (vectors['x0'] < 4.8e4)
mask = mask & (vectors['y0'] >= 6.43e4) & (vectors['y0'] < 6.53e4)
# project vertical components of polarization angles onto the data
ax = sc.pl.spatial(adata[mask], spot_size=0, show=False)
ax = sc.pl.spatial(adata[mask], spot_size=30, ax=ax[0], color=['unit_v'], vmin=-1, vmax=1, cmap='coolwarm')

# retrieve expression and 
df_gex = sc.get.obs_df(adata[mask], keys=adata.var.index.tolist()+['unit_v']).dropna()
df_gex = df_gex.loc[:, df_gex.nunique(0) > 1]
df_gex = df_gex.loc[df_gex.iloc[:, :-1].sum(1) > 0]
# compute correlations (top 100 were utilized for correlation)
df_corr = pd.DataFrame(columns=['rho','pval'])
for gene in tqdm(df_gex.columns[:-1]):
    df_corr.loc[gene] = ss.pearsonr(df_gex[gene], df_gex['unit_v'])
df_corr.to_csv('/PATH/TO/SI_TABLES/Table_S1.csv')
# ADATA object has been deposited in Zenodo

# define the pathways of interest (complete list in Table S2)
paths = '''Skeletal System Development (GO:0001501)
Cartilage Development (GO:0051216)
Chondrocyte Differentiation (GO:0002062)
Canonical Wnt Signaling Pathway (GO:0060070)
Positive Regulation of Cell Differentiation (GO:0045597)
Skin Development (GO:0043588)
Wnt Signaling Pathway (GO:0016055)
Mitotic Spindle Assembly (GO:0090307)'''.split('\n')
data = pd.read_excel('/PATH/TO/SI_TABLES/Table_S2.xlsx', index_col=0)

# select the values of interest
ors = data.loc[paths, 'Odds Ratio']
vals = data.loc[paths, 'Adjusted P-value']

# retrieve colors and plot the bas
from matplotlib.cm import get_cmap
from matplotlib.colors import to_hex
cmap = get_cmap('Reds')
colors = [to_hex(cmap(el / 20)) for el in ors]
fig, ax = plt.subplots(figsize=[4, 3.5])
_ = ax.grid(False)
_ = ax.bar(vals.index, -np.log10(vals), color=colors, edgecolor='k')
_ = ax.tick_params(axis='x', labelrotation=90)

### Fig. S3 (Colonocyte Validation)

In [ ]:
# Read in the segmentation and marker intensities file (derived from pipeline offset)
# > Define file name
segmentation_file = '/PATH/TO/CODEX_HICKEY_ET_AL/HBM447.MNKW.592/reg001_mask.ome.tiff'
intensities_file = '/PATH/TO/CODEX_HICKEY_ET_AL/HBM447.MNKW.592/reg001_expr.ome.tiff'
# > Process data in files
segmentation_imstack, marker_imstack = load_data(segmentation_file, intensities_file)
segmentation_dict = create_segmentation_dict(segmentation_imstack[0, 0, :, :])
# > Define marker of interest and locate in image stack
marker_of_interest = 'Vimentin'
marker_list = create_marker_list_xeniumcodex(intensities_file)
image_stack_index = marker_list.index(marker_of_interest)
# > Retrieve intensities for the marker of interest
intensities = marker_imstack[0][image_stack_index]
# > Set accelerator device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
intensities_acc = torch.tensor(intensities, device=device, dtype=torch.float32)

# Generate smoothed vectors
vectors = generate_vector_field(segmentation_dict, intensities_acc[0, :, :], device)
smoothed_vectors = smooth_vector_field(vectors,
                                       min_magnitude=MIN_VECTOR_MAGNITUDE,
                                       smooth_iters=SMOOTH_ITERS,
                                       smooth_dist_limit_pixel=SMOOTH_DIST_LIMIT_PIXEL)
# Calculate curl and divergence metrics
XI, YI, curl, divergence = calculate_curl_and_divergence(smoothed_vectors)

# custom plotting
# setup matplotlib figure
fig, ax = plt.subplots(figsize=FIGSIZE_PLOT)
ax.grid(False)
# plot quivers
SCALE = 0.25
ax.scatter(smoothed_vectors['x0'], smoothed_vectors['y0'], color=CENTROID_COLOR, s=CENTROID_SIZE)
ax.quiver(smoothed_vectors['x0'], smoothed_vectors['y0'], smoothed_vectors['unit_u'], smoothed_vectors['unit_v'],
          width=QUIVER_WIDTH * SCALE, scale=QUIVER_SCALE / SCALE)

# custom plotting for a masked region
mask = (smoothed_vectors['x0'] >= 2000) & (smoothed_vectors['x0'] < 4000)
mask = mask & (smoothed_vectors['y0'] >= 7000) & (smoothed_vectors['y0'] < 9000)
# setup matplotlib figure
SCALE = 1
fig, ax = plt.subplots(figsize=FIGSIZE_PLOT); ax.grid(False)
ax.scatter(smoothed_vectors['x0'][mask], smoothed_vectors['y0'][mask],
           color=CENTROID_COLOR, s=CENTROID_SIZE * 10)
ax.quiver(smoothed_vectors['x0'][mask], smoothed_vectors['y0'][mask],
          smoothed_vectors['unit_u'][mask], smoothed_vectors['unit_v'][mask],
          width=QUIVER_WIDTH * SCALE, scale=QUIVER_SCALE / SCALE)

### Fig. 3 and S4 (Melanoma Cytoskeleton Dynamic Phenotypes)

In [ ]:
# Read in the segmentation and marker intensities file
# > Define file name
segmentation_file = '/PATH/TO/PT0711_BASELINE_SEGMENTATION_TIFF'
intensities_file = '/PATH/TO/PT0711_BASELINE_MIBI_INTENSITIES_TIFF'
# > Process data in files
segmentation_imstack, marker_imstack = load_data(segmentation_file, intensities_file)
segmentation_dict = create_segmentation_dict(segmentation_imstack)
# > Define marker of interest and locate in image stack
marker_of_interest = 'Vimentin'
marker_list = create_marker_list_mibiv2(intensities_file)
image_stack_index = marker_list.index(marker_of_interest)
# > Retrieve intensities for the marker of interest
intensities = marker_imstack[0][image_stack_index]
# > Set accelerator device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
intensities_acc = torch.tensor(intensities, device=device, dtype=torch.float32)

# Generate smoothed vectors
vectors = generate_vector_field(segmentation_dict, intensities_acc, device)
smoothed_vectors = smooth_vector_field(vectors,
                                       min_magnitude=MIN_VECTOR_MAGNITUDE,
                                       smooth_iters=SMOOTH_ITERS,
                                       smooth_dist_limit_pixel=SMOOTH_DIST_LIMIT_PIXEL)

import anndata as ad
# read in the expression
X = pd.read_table('/PATH/TO/PT0711_BASELINE_MIBI_INTENSITIES_TSV')
X = X.pivot_table(index='segment', columns='marker', values='sum_segmentint').fillna(0)
# create an anndata object for the data
adata = ad.AnnData(X)
# read in the cell types
cell_types = pd.read_csv('/PATH/TO/PT0711_BASELINE_CELL_TYPES_CSV', index_col=0)
cell_types.index = cell_types.index.astype(str)
adata.obs['cell_type'] = cell_types
# add in raw unscaled spatial
spatial = vectors.set_index('segment')[['x0','y0','magnitude']]
spatial.index = spatial.index.astype(str)
adata.obs[spatial.columns] = spatial
adata.obsm['spatial'] = adata.obs[['x0','y0']].values
# add on smoothed vectors
movement = smoothed_vectors.set_index('segment')[['unit_u','unit_v']]
movement.index = movement.index.astype(str)
adata.obs[movement.columns] = movement
# normalize data
adata.layers['raw_counts'] = adata.X.copy()
adata.X = np.log10(adata.X + 1)
# convert the cell types into the same categories
categories = ['B cell', 'CD4 T cell', 'CD8 T cell', 'Conflicted', 'Dendritic',
       'Endothelial', 'Fibroblast', 'Immune', 'Macrophage', 'Monocyte',
       'Myeloid', 'NK cell', 'Neutrophil', 'Plasma cell', 'T cell', 'Treg', 'Tumor', 'Unknown']
adata.obs['cell_type'] = pd.Categorical(adata.obs['cell_type'], categories=categories)

# project angles onto the data
fig, ax = plt.subplots(figsize=[6, 6])
ax = sc.pl.spatial(adata, spot_size=0, show=False, ax=ax)
ax = sc.pl.spatial(adata, spot_size=20, ax=ax[0], color=['cell_type'], show=False, alpha=0.8, title='Cell Type')[0]
ax.quiver(adata.obs['x0'], adata.obs['y0'], adata.obs['unit_u'], adata.obs['unit_v'],
          width=QUIVER_WIDTH * 1, scale=QUIVER_SCALE / 1, alpha=0.8)

In [ ]:
# Read in the segmentation and marker intensities file
# > Define file name
segmentation_file = '/PATH/TO/PT0613_BASELINE_SEGMENTATION_TIFF'
intensities_file = '/PATH/TO/PT0613_BASELINE_MIBI_INTENSITIES_TIFF'
# > Process data in files
segmentation_imstack, marker_imstack = load_data(segmentation_file, intensities_file)
segmentation_dict = create_segmentation_dict(segmentation_imstack)
# > Define marker of interest and locate in image stack
marker_of_interest = 'Vimentin'
marker_list = create_marker_list_mibiv2(intensities_file)
image_stack_index = marker_list.index(marker_of_interest)
# > Retrieve intensities for the marker of interest
intensities = marker_imstack[0][image_stack_index]
# > Set accelerator device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
intensities_acc = torch.tensor(intensities, device=device, dtype=torch.float32)

# Generate smoothed vectors
vectors = generate_vector_field(segmentation_dict, intensities_acc, device)
smoothed_vectors = smooth_vector_field(vectors,
                                       min_magnitude=MIN_VECTOR_MAGNITUDE,
                                       smooth_iters=SMOOTH_ITERS,
                                       smooth_dist_limit_pixel=SMOOTH_DIST_LIMIT_PIXEL)

# read in the expression
X = pd.read_table('/PATH/TO/PT0613_BASELINE_MIBI_INTENSITIES_TSV')
X = X.pivot_table(index='segment', columns='marker', values='sum_segmentint').fillna(0)
# create an anndata object for the data
adata = ad.AnnData(X)
# read in the cell types
cell_types = pd.read_csv('/PATH/TO/PT0613_BASELINE_CELL_TYPES_CSV', index_col=0)
cell_types.index = cell_types.index.astype(str)
adata.obs['cell_type'] = cell_types
# add in raw unscaled spatial
spatial = vectors.set_index('segment')[['x0','y0','magnitude']]
spatial.index = spatial.index.astype(str)
adata.obs[spatial.columns] = spatial
adata.obsm['spatial'] = adata.obs[['x0','y0']].values
# add on smoothed vectors
movement = smoothed_vectors.set_index('segment')[['unit_u','unit_v']]
movement.index = movement.index.astype(str)
adata.obs[movement.columns] = movement
# normalize data
adata.layers['raw_counts'] = adata.X.copy()
adata.X = np.log10(adata.X + 1)
# convert the cell types into the same categories
categories = ['B cell', 'CD4 T cell', 'CD8 T cell', 'Conflicted', 'Dendritic',
       'Endothelial', 'Fibroblast', 'Immune', 'Macrophage', 'Monocyte',
       'Myeloid', 'NK cell', 'Neutrophil', 'Plasma cell', 'T cell', 'Treg', 'Tumor', 'Unknown']
adata.obs['cell_type'] = pd.Categorical(adata.obs['cell_type'], categories=categories)

# project angles onto the data
fig, ax = plt.subplots(figsize=[6, 6])
ax = sc.pl.spatial(adata, spot_size=0, show=False, ax=ax)
ax = sc.pl.spatial(adata, spot_size=20, ax=ax[0], color=['cell_type'], show=False, alpha=0.8, title='Cell Type')[0]
ax.quiver(adata.obs['x0'], adata.obs['y0'], adata.obs['unit_u'], adata.obs['unit_v'],
          width=QUIVER_WIDTH * 1, scale=QUIVER_SCALE / 1, alpha=0.8)

In [ ]:
# Read in the segmentation and marker intensities file
# > Define file name
segmentation_file = '/PATH/TO/PT0679_BASELINE_SEGMENTATION_TIFF'
intensities_file = '/PATH/TO/PT0679_BASELINE_MIBI_INTENSITIES_TIFF'
# > Process data in files
segmentation_imstack, marker_imstack = load_data(segmentation_file, intensities_file)
segmentation_dict = create_segmentation_dict(segmentation_imstack)
# > Define marker of interest and locate in image stack
marker_of_interest = 'Vimentin'
marker_list = create_marker_list_mibiv2(intensities_file)
image_stack_index = marker_list.index(marker_of_interest)
# > Retrieve intensities for the marker of interest
intensities = marker_imstack[0][image_stack_index]
# > Set accelerator device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
intensities_acc = torch.tensor(intensities, device=device, dtype=torch.float32)

# Generate smoothed vectors
vectors = generate_vector_field(segmentation_dict, intensities_acc, device)
smoothed_vectors = smooth_vector_field(vectors,
                                       min_magnitude=MIN_VECTOR_MAGNITUDE,
                                       smooth_iters=SMOOTH_ITERS,
                                       smooth_dist_limit_pixel=SMOOTH_DIST_LIMIT_PIXEL)

import anndata as ad
# read in the expression
X = pd.read_table('/PATH/TO/PT0679_BASELINE_MIBI_INTENSITIES_TSV')
X = X.pivot_table(index='segment', columns='marker', values='sum_segmentint').fillna(0)
# create an anndata object for the data
adata = ad.AnnData(X)
# read in the cell types
cell_types = pd.read_csv('/PATH/TO/PT0679_BASELINE_CELL_TYPES_CSV', index_col=0)
cell_types.index = cell_types.index.astype(str)
adata.obs['cell_type'] = cell_types
# add in raw unscaled spatial
spatial = vectors.set_index('segment')[['x0','y0','magnitude']]
spatial.index = spatial.index.astype(str)
adata.obs[spatial.columns] = spatial
adata.obsm['spatial'] = adata.obs[['x0','y0']].values
# add on smoothed vectors
movement = smoothed_vectors.set_index('segment')[['unit_u','unit_v']]
movement.index = movement.index.astype(str)
adata.obs[movement.columns] = movement
# normalize data
adata.layers['raw_counts'] = adata.X.copy()
adata.X = np.log10(adata.X + 1)
# convert the cell types into the same categories
categories = ['B cell', 'CD4 T cell', 'CD8 T cell', 'Conflicted', 'Dendritic',
       'Endothelial', 'Fibroblast', 'Immune', 'Macrophage', 'Monocyte',
       'Myeloid', 'NK cell', 'Neutrophil', 'Plasma cell', 'T cell', 'Treg', 'Tumor', 'Unknown']
adata.obs['cell_type'] = pd.Categorical(adata.obs['cell_type'], categories=categories)

# project angles onto the data
fig, ax = plt.subplots(figsize=[6, 6])
ax = sc.pl.spatial(adata, spot_size=0, show=False, ax=ax)
ax = sc.pl.spatial(adata, spot_size=20, ax=ax[0], color=['cell_type'], show=False, alpha=0.8, title='Cell Type')[0]
ax.quiver(adata.obs['x0'], adata.obs['y0'], adata.obs['unit_u'], adata.obs['unit_v'],
          width=QUIVER_WIDTH * 1, scale=QUIVER_SCALE / 1, alpha=0.8)

In [ ]:
from tqdm import tqdm
from glob import glob
# Gather all files
fns = glob('/PATH/TO/MELANOMA_BIOPSY_TIFFS')
Xs = []
for fn in tqdm(fns):
    try:
        # Add in special naming
        dirname1 = fn.split('/')[-2]
        dirname2 = dirname1[:6] + '_' + dirname1[7:]
        vectors = fn2vector[fn]
        smoothed_vectors = smooth_vector_field(vectors.iloc[:, :-1],
                                               min_magnitude=MIN_VECTOR_MAGNITUDE,
                                               smooth_iters=SMOOTH_ITERS,
                                               smooth_dist_limit_pixel=SMOOTH_DIST_LIMIT_PIXEL)
        XI, YI, curl, divergence = calculate_curl_and_divergence(smoothed_vectors)
        X = pd.read_table(f'../MORRISON/morrison-cohort-3-mibi/intensities/{dirname2}.intensities.tsv')
        X = X.pivot_table(index='segment', columns='marker', values='sum_segmentint').fillna(0)
        
        # compile the curl and divergence metrics into a dataframe
        df = pd.DataFrame([XI.flatten(), YI.flatten(), curl.flatten(), divergence.flatten()]).T.dropna()
        df.columns = ['x','y','curl','divergence']
        # get distance matrix from cells to x, y
        dist = cdist(vectors[['x0','y0']], df[['x','y']]) * 0.390625
        # mask out >=100µm connections
        dist_masked = dist.copy()
        dist_masked[dist >= 100] = 0
        # log10p1 for the weight
        dist_masked = np.log10(dist_masked + 1)
        weights = (dist_masked.T / dist_masked.sum(1)).T
        # expand out the curl and divergence to the vectors
        vectors[['curl','divergence']] = (weights @ df[['curl','divergence']]).values
        vectors['abs_curl'] = vectors['curl'].abs()
        vectors['abs_divergence'] = vectors['divergence'].abs()
    
        # realign the matrices
        idxs = vectors['segment'][vectors['segment'].isin(X.index)]
        X = X.loc[idxs]
        vectors = vectors.set_index('segment').loc[idxs]
        X = pd.concat([X, vectors], axis=1)
        X['sample'] = dirname1
        Xs.append(X)
    except:
        pass

# concatenate the data together
X = pd.concat(Xs, axis=0).reset_index()
genes = X.columns[1:34]
samples = X['sample'].unique()
# exclude the cell types that are not of interest (e.g. rare or unknown)
cts = [[el] for el in X['celltype'].unique() if el not in ['NKT cell','T cell','Immune','Unknown','Conflicted']]
# add on the "All" cell type which is a list of all cell types (this is our "bulk" sample)
cts.append(list(X['celltype'].unique()))

In [ ]:
# read in the annotations
annos1 = pd.read_excel('../MORRISON/morrison-cohort-3-mibi/MORRISON_ptII_Cohort_3_Annotation.xlsx', sheet_name=1)
annos1_subset = annos1.set_index('label')[['participant','Arm','_group','Timepoint']]
annos1_subset.index = annos1_subset.index.str.replace('-','_')
annos1_subset['Arm_Timepoint'] = annos1_subset[['Arm','Timepoint']].agg(':'.join, axis=1)
annos1_subset['Arm_Group'] = annos1_subset[['Arm','_group']].agg(':'.join, axis=1)
# rename the annotations appropriately
annos1_subset['Group'] = annos1_subset['Arm_Group'].str.replace('Nivolumab + Ipilimumab:S','N+I:R').str.replace('Nivolumab + Ipilimumab:D','N+I:NR')
annos1_subset.loc[~annos1_subset['Group'].str.startswith('N+I'), 'Group'] = 'I:NR'
annos1_subset['Timepoint'] = annos1_subset['Timepoint'].str.replace('OnTx- C2','On Therapy')
annos1_subset.index = annos1_subset.index.str.replace('_','-')
annos1_subset['Timepoint'] = annos1_subset['Timepoint'].str.replace('On Therapy','TX').str.replace('Screening','BL')

# compute percentiles for each sample
for stat in ['magnitude','curl','divergence','abs_curl','abs_divergence']:
    X[stat+'_PERC'] = np.nan
    for sample in tqdm(samples):
        mask_s = X['sample'] == sample
        vs = X.loc[mask_s, stat]
        X.loc[mask_s, stat+'_PERC'] = [ss.percentileofscore(vs, el) for el in vs]

# compute the statistics for robustly represented samples (e.g. ≥2)
stats = pd.DataFrame(columns=['sample','celltype','stat','gene','rho','pval'])
for sample in tqdm(samples):
    mask_s = X['sample'] == sample
    for ct in cts:
        mask_ct = X['celltype'].isin(ct)
        if sum(mask_s & mask_ct) < 2: continue
        ct_label = ct[0] if len(ct) == 1 else 'All'
        for gene in genes:
            v1s = X.loc[mask_s & mask_ct, gene]
            if v1s.nunique() < 2: continue
            for stat in ['magnitude','curl','divergence','abs_curl','abs_divergence']:
                v2s = X.loc[mask_s & mask_ct, stat+'_PERC'].dropna().round()
                if v2s.nunique() < 2: continue
                vs = pd.concat([v1s, v2s], axis=1).groupby(stat+'_PERC').mean(numeric_only=True)[gene]
                rho, pval = ss.pearsonr(vs.index, vs)
                stats.loc[stats.shape[0]] = sample, ct_label, stat, gene, rho, pval

# add on the different response types
stats['sample_wo_roi'] = stats['sample'].str.split('_ROI', expand=True).iloc[:, 0]
stats['arm_resp'] = stats['sample_wo_roi'].map(annos1_subset['Group']).values
stats['time'] = stats['sample_wo_roi'].map(annos1_subset['Timepoint']).values
stats['arm_resp_time'] = stats[['arm_resp','time']].astype(str).agg(' @ '.join, axis=1)

In [ ]:
# examine the overall statistics (i.e. cytoskeleton dynamics x phenotype [protein])
mask = (stats['celltype'] == 'All')
data = stats.loc[mask].copy()
data['gene'] = data['gene'].str.replace('HLA class 1 A, B, and C, Na-K-ATPase','HLA ABC')
data_rho = data.pivot_table(index='stat', columns='gene', values='rho')
data_pval = data.pivot_table(index='stat', columns='gene', values='pval')
# look at the correlation across data
g = sns.clustermap(data_rho, vmin=-0.2, vmax=0.2, method='ward', cmap='coolwarm',
                   figsize=[8, 3.5], xticklabels=1, yticklabels=1, dendrogram_ratio=(.05, .2),
                   cbar_pos=(-0.02, .9, .009, .11))
_ = g.ax_heatmap.grid(False)
_ = g.ax_cbar.tick_params(axis='y', labelrotation=180)
_ = g.ax_heatmap.tick_params(axis='y', labelrotation=180)
_ = g.ax_heatmap.set(xlabel=None, ylabel=None)

# report on all cell type specific relationships of the above
for stat in ['magnitude','curl','divergence','abs_curl','abs_divergence']:
    mask = (stats['celltype'] != 'All') & (stats['stat'] == stat)
    data = stats.loc[mask].copy()
    data['gene'] = data['gene'].str.replace('HLA class 1 A, B, and C, Na-K-ATPase','HLA ABC')
    data_rho = data.pivot_table(index='celltype', columns='gene', values='rho')
    data_pval = data.pivot_table(index='celltype', columns='gene', values='pval')
    # report the heatmap
    import seaborn as sns
    g = sns.clustermap(data_rho, method='ward', vmin=-0.2, vmax=0.2, cmap='coolwarm',
                       figsize=[6.5, 4.5], xticklabels=1, yticklabels=1,
                       dendrogram_ratio=(.1, .35), cbar_pos=(0, .86, .01, .09))
    _ = g.ax_heatmap.grid(False); _ = g.ax_heatmap.set_facecolor('lightgray')
    _ = g.ax_heatmap.set(xlabel='Correlation w/ '+stat.replace('_',' ').title(), ylabel=None)

In [ ]:
# examine the correlations with divergence for all cells
for gene in ['CD3','LAG3','TIM-3']:
    # read in the data
    mask = (stats['celltype'] == 'All') & (stats['stat'] == 'divergence') & (stats['gene'] == gene)
    data = stats.loc[mask].copy()
    p1 = ss.mannwhitneyu(data.loc[data['arm_resp_time'] == 'N+I:R @ BL', 'rho'], data.loc[data['arm_resp_time'] == 'N+I:R @ TX', 'rho'])[1]
    p2 = ss.mannwhitneyu(data.loc[data['arm_resp_time'] == 'N+I:R @ BL', 'rho'], data.loc[data['arm_resp_time'] == 'N+I:NR @ BL', 'rho'])[1]
    valid = (p1 < 0.05) | (p2 < 0.05)
    if not valid: continue
    print(gene, p1, p2)
    # perform the genes
    fig, ax = plt.subplots(figsize=[4, 4])
    _ = ax.grid(False)
    sns.barplot(x='arm_resp', hue='time', y='rho', data=data, ci=95, errwidth=1.5, capsize=0.3,
                errcolor='k', linewidth=1.5, edgecolor='k', hue_order=['BL','TX'], ax=ax,
                order=['N+I:R','N+I:NR','I:NR'], palette=['skyblue','dodgerblue'], saturation=1)
    _ = ax.axhline(0, color='k', linewidth=1.5)
    _ = ax.set_xlim(-1, 3)
    _ = ax.legend(frameon=False, title='Timepoint', bbox_to_anchor=(1, .5), bbox_transform=ax.transAxes, loc='center left')
    _ = ax.set(xlabel=None, ylabel=f'Correlation w/ {gene} (Rho)')

# repeat the analysis for CD8+ T cells
for gene in ['Ki67','LAG3','TIM-3']:
    # read in the data
    mask = (stats['celltype'] == 'CD8 T cell') & (stats['stat'] == 'divergence') & (stats['gene'] == gene)
    data = stats.loc[mask].copy()
    p1 = ss.mannwhitneyu(data.loc[data['arm_resp_time'] == 'N+I:R @ BL', 'rho'], data.loc[data['arm_resp_time'] == 'N+I:R @ TX', 'rho'])[1]
    p2 = ss.mannwhitneyu(data.loc[data['arm_resp_time'] == 'N+I:R @ BL', 'rho'], data.loc[data['arm_resp_time'] == 'N+I:NR @ BL', 'rho'])[1]
    valid = (p1 < 0.05) | (p2 < 0.05)
    if not valid: continue
    print(gene, p1, p2)
    # perform the genes
    fig, ax = plt.subplots(figsize=[4, 4])
    _ = ax.grid(False)
    sns.barplot(x='arm_resp', hue='time', y='rho', data=data, ci=95, errwidth=1.5, capsize=0.3,
                errcolor='k', linewidth=1.5, edgecolor='k', hue_order=['BL','TX'], ax=ax,
                order=['N+I:R','N+I:NR','I:NR'], palette=['skyblue','dodgerblue'], saturation=1)
    _ = ax.axhline(0, color='k', linewidth=1.5)
    _ = ax.set_xlim(-1, 3)
    _ = ax.legend(frameon=False, title='Timepoint', bbox_to_anchor=(1, .5), bbox_transform=ax.transAxes, loc='center left')
    _ = ax.set(xlabel=None, ylabel=f'Correlation w/ {gene} (Rho)')

In [ ]:
# plot the averages for magnitude
avgs = X[['sample','magnitude']].groupby('sample').mean()
avgs['sample_wo_roi'] = avgs.index.to_series().str.split('_ROI', expand=True).iloc[:, 0]
avgs['arm_resp'] = avgs['sample_wo_roi'].map(annos1_subset['Group']).values
avgs['time'] = avgs['sample_wo_roi'].map(annos1_subset['Timepoint']).values
avgs['arm_resp_time'] = avgs[['arm_resp','time']].astype(str).agg(' @ '.join, axis=1)

fig, ax = plt.subplots(figsize=[4, 4])
_ = ax.grid(False)
sns.boxplot(x='arm_resp', hue='time', y='magnitude', data=avgs, linewidth=1.5, linecolor='k',
            hue_order=['BL','TX'], ax=ax, order=['N+I:R','N+I:NR','I:NR'], palette=['skyblue','dodgerblue'], saturation=1)
_ = ax.set_xlim(-1, 3)
_ = ax.legend(frameon=False, title='Timepoint', bbox_to_anchor=(1, .5), bbox_transform=ax.transAxes, loc='center left')
_ = ax.set(xlabel=None, ylabel='Magnitude')

### Fig. 4 and S5 (Clustering of Regions with Similar Cytoskeleton Dynamics)

In [ ]:
# perform DBSCAN to cluster the samples based on the curl and divergence ONLY (we avoid directionality)
def cluster_data(data, eps=0.4, min_samples=10):
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    clusters = dbscan.fit_predict(data.values)
    clusters = pd.Series(clusters, index=data.index)
    return clusters
    
# define the samples to examine
samples = X['sample'].unique()
# instantiate a column to track the cluster assignments
X['cd_cluster'] = np.nan
# loop through each sample
for sample1 in samples:
    # extract the correct sample name
    start = sample1.index('-')
    start += sample1[start+1:].index('-')+1
    data = X.loc[X['sample'] == sample1].copy()
    # normalize the position and curl/divergence metrics
    data[['x0','y0']] -= data[['x0','y0']].mean()
    data[['x0','y0']] /= data[['x0','y0']].std()
    data[['curl','divergence']] /= 0.001
    # cluster the data by the cytoskeleton dynamics and spatial positions
    clusters = cluster_data(data[['x0','y0','curl','divergence']].dropna()).reindex(data.index).fillna(-1)
    X.loc[data.index, 'cd_cluster'] = data['cluster'] = clusters

# define a mask for regions that are not background (i.e. not -1)
X['sample_cluster'] = X['sample'] + ':' + X['cd_cluster'].astype(int).astype(str)
mask = X['cd_cluster'] > -1

# aggregate data into a regional anndata object and normalize the vlaues
import anndata
data = X.loc[mask, genes + ['abs_curl','divergence','sample_cluster']].groupby('sample_cluster').mean()
ad = anndata.AnnData(data)
ad.raw = ad.copy()
ad.X -= ad.X.mean(axis=0); ad.X /= ad.X.std(axis=0)
# perform a kNN weighted towards cytoskeleton dynamics
scale = 6
ad[:, ad.var.index[:-2]].X *= scale / len(ad.var.index[:-2])
# compute dimension reduction
sc.tl.pca(ad)
sc.pp.neighbors(ad, random_state=0)
sc.tl.umap(ad, random_state=0)
# project the clusters
ad.obs[['sample','cluster']] = ad.obs.index.to_series().str.split(':', expand=True)
# compute leiden
sc.tl.leiden(ad, flavor='igraph', n_iterations=2, random_state=0, resolution=0.9)
sc.pl.umap(ad, color=['leiden'])
# reset the values post computing the equal weighted projection and clustering
ad[:, ad.var.index[:-2]].X /= scale / len(ad.var.index[:-2])

# retrieve the protein expression values per regional cluster
data = sc.get.obs_df(ad, keys=ad.var.index.tolist()+['leiden']).groupby('leiden').mean()
data.columns = data.columns.str.replace('HLA class 1 A, B, and C, Na-K-ATPase','HLA ABC')
g = sns.clustermap(data, method='ward', vmin=-1, vmax=1, cmap='coolwarm', cbar_pos=(0, .9, .01, .08),
                   figsize=[10, 7.5], dendrogram_ratio=(.1, .32), xticklabels=1, yticklabels=1)
_ = g.ax_heatmap.grid(False)

In [ ]:
# define a mask for regions that are not background (i.e. not -1)
X['sample_cluster'] = X['sample'] + ':' + X['cd_cluster'].astype(int).astype(str)
mask = X['cd_cluster'] > -1
# subset for the regional leiden of interest ('0')
valids = ad.obs.index[ad.obs['leiden'] == '0']
counts = X['sample_cluster'].value_counts().loc[valids].sort_values()[::-1]

# list out the categories (cell types)
cats = ['B cell', 'CD4 T cell', 'CD8 T cell', 'Conflicted', 'Dendritic',
        'Endothelial', 'Fibroblast', 'Immune', 'Macrophage', 'Monocyte',
        'Myeloid', 'NK cell', 'Neutrophil', 'Plasma cell', 'T cell', 'Treg', 'Tumor', 'Unknown']

# compare the distribution of leidens
counts_set = X.loc[X['sample_cluster'].isin(counts.index), 'celltype'].value_counts().reindex(cats).fillna(0)
counts_all = X.loc[~X['sample_cluster'].isin(counts.index), 'celltype'].value_counts().reindex(cats).fillna(0)
fig, ax = plt.subplots(); _ = ax.pie(counts_all, colors=sc.pl.palettes.default_20)
fig, ax = plt.subplots(); _ = ax.pie(counts_set, colors=sc.pl.palettes.default_20)
# compute p-value
import scipy.stats as ss
p = ss.chi2_contingency(pd.concat([counts_set, counts_all], axis=1)).pvalue
print(p)

# evaluate for any enrichment of T cells
diff = (counts_set / counts_set.sum() * 100) - (counts_all / counts_all.sum() * 100)
# read in the data
fig, ax = plt.subplots()
_ = ax.grid(False)
_ = ax.bar(diff.index, diff, color='skyblue', edgecolor='dodgerblue', linewidth=1.5)
_ = ax.axhline(0, color='k', linewidth=1.5)
_ = ax.tick_params(axis='x', labelrotation=90)
_ = ax.set(xlabel='Leiden', ylabel='Percentage (%) Difference\nNear and Far to Streams')

In [ ]:
# plot regional cluster assignment for a given biopsy
sample, cluster = sample_cluster.split(':')
data = X.loc[X['sample'] == sample].copy()
# plot the clusters
fig, ax = plt.subplots(figsize=[6, 6])
_ = ax.grid(False)
clusters = data['cd_cluster']
for cluster in sorted(clusters.unique()):
    if cluster == -1: continue
    mask = clusters == cluster
    ax.scatter(data.loc[mask, 'x0'], data.loc[mask, 'y0'], s=10, label=cluster, edgecolor='k', linewidth=0.2, alpha=0.8)
mask = clusters == -1
ax.scatter(data.loc[mask, 'x0'], data.loc[mask, 'y0'], s=2, color='lightgray', label='NA')
_ = ax.set_xticks([]); _ = ax.set_yticks([])
_ = ax.legend(bbox_to_anchor=(1, .5), bbox_transform=ax.transAxes, markerscale=2,
              loc='center left', ncols=2, frameon=False, columnspacing=0.3, title='Clusters')
_ = ax.set_title(sample)
# smooth the vectors
vectors = X.loc[X['sample'] == sample, ['x0','y0','xv','yv','segment','magnitude','celltype','sample_cluster']].copy()
svectors = smooth_vector_field(vectors.iloc[:, :-2],
                               min_magnitude=MIN_VECTOR_MAGNITUDE,
                               smooth_iters=SMOOTH_ITERS,
                               smooth_dist_limit_pixel=SMOOTH_DIST_LIMIT_PIXEL)
svectors = svectors.loc[svectors.index.intersection(vectors.index[vectors['sample_cluster'] == sample_cluster])].copy()
_ = ax.quiver(svectors['x0'], svectors['y0'], svectors['unit_u'], svectors['unit_v'],
              width=QUIVER_WIDTH * 2, scale=QUIVER_SCALE / 1.5, alpha=0.8)

In [ ]:
# plot cell type assignment for a given biopsy
sample, cluster_chosen = sample_cluster.split(':')
data = X.loc[X['sample'] == sample].copy()
# plot the clusters
fig, ax = plt.subplots(figsize=[6, 6])
_ = ax.grid(False)
mask = data['cd_cluster'].astype(float).astype(str) == str(float(cluster_chosen))
for ct, color in zip(cats, colors):
    mask_ct = data['celltype'] == ct
    ax.scatter(data.loc[(mask)&mask_ct, 'x0'], data.loc[(mask)&mask_ct, 'y0'], color=color, s=10, label=ct, edgecolor='k', linewidth=0.2, alpha=0.8)
    ax.scatter(data.loc[(~mask)&mask_ct, 'x0'], data.loc[(~mask)&mask_ct, 'y0'], color=color, s=4, alpha=0.5)
_ = ax.set_xticks([]); _ = ax.set_yticks([])
_ = ax.legend(bbox_to_anchor=(1, .5), bbox_transform=ax.transAxes, markerscale=2,
              loc='center left', ncols=2, frameon=False, columnspacing=0.3, title='Cell Type')
_ = ax.set_title(sample)
# smooth the vectors
vectors = X.loc[X['sample'] == sample, ['x0','y0','xv','yv','segment','magnitude','celltype','sample_cluster']].copy()
svectors = smooth_vector_field(vectors.iloc[:, :-2],
                               min_magnitude=MIN_VECTOR_MAGNITUDE,
                               smooth_iters=SMOOTH_ITERS,
                               smooth_dist_limit_pixel=SMOOTH_DIST_LIMIT_PIXEL)
svectors = svectors.loc[svectors.index.intersection(vectors.index[vectors['sample_cluster'] == sample_cluster])].copy()
_ = ax.quiver(svectors['x0'], svectors['y0'], svectors['unit_u'], svectors['unit_v'],
              width=QUIVER_WIDTH * 1.5, scale=QUIVER_SCALE / 1, alpha=0.8)

In [ ]:
# examine for the relationship between leiden 0 and response
X['sample_cluster_leiden'] = X['sample_cluster'].map(ad.obs['leiden'])
counts = X.loc[X['sample_cluster_leiden'] == '0', 'sample'].value_counts()
counts /= X['sample'].value_counts(); counts *= 100
data = counts.reset_index()
data['sample_wo_roi'] = data['sample'].str.split('_ROI', expand=True).iloc[:, 0]
data['sample_wo_roi'] = data['sample'].str.split('_', expand=True).iloc[:, 0]
data['arm_resp'] = data['sample_wo_roi'].map(annos1_subset['Group']).values
data['time'] = data['sample_wo_roi'].map(annos1_subset['Timepoint']).values
data['arm_resp_time'] = data[['arm_resp','time']].astype(str).agg(' @ '.join, axis=1)

# perform non-parametric statistical tests
p1 = ss.mannwhitneyu(data.loc[data['arm_resp_time'] == 'N+I:R @ BL', 'count'],
                     data.loc[data['arm_resp_time'] == 'N+I:R @ TX', 'count'])[1]
p2 = ss.mannwhitneyu(data.loc[data['arm_resp_time'] == 'N+I:R @ BL', 'count'],
                     data.loc[data['arm_resp_time'] == 'N+I:NR @ BL', 'count'])[1]
# plot the distribution on a bar plot
fig, ax = plt.subplots(figsize=[4, 4])
_ = ax.grid(False)
sns.barplot(x='arm_resp', hue='time', y='count', data=data, ci=95, errwidth=1.5, capsize=0.3,
            errcolor='k', linewidth=1.5, edgecolor='k', hue_order=['BL','TX'], ax=ax,
            order=['N+I:R','N+I:NR','I:NR'], palette=['skyblue','dodgerblue'], saturation=1)
_ = ax.axhline(0, color='k', linewidth=1.5)
_ = ax.set_xlim(-1, 3)
_ = ax.legend(frameon=False, title='Timepoint', bbox_to_anchor=(1, .5), bbox_transform=ax.transAxes, loc='center left')
_ = ax.set(xlabel=None, ylabel='Region Percentage (%)')

### Fig. 5 and S6 ("Streams" in a Biopsy of Melanoma)

In [ ]:
# read in the expression
X = pd.read_table('/PATH/TO/PT0242_ONTX_BIOPSY_INTENSITIES_TSV')
X = X.pivot_table(index='segment', columns='marker', values='sum_segmentint').fillna(0)
# create an anndata object for the data
adata = ad.AnnData(X)
# read in the cell types
cell_types = pd.read_csv('/PATH/TO/PT0242_ONTX_BIOPSY_CELL_TYPES', index_col=0)
cell_types.index = cell_types.index.astype(str)
adata.obs['cell_type'] = cell_types
# add in raw unscaled spatial
spatial = vectors.set_index('segment')[['x0','y0','magnitude']]
spatial.index = spatial.index.astype(str)
adata.obs[spatial.columns] = spatial
adata.obsm['spatial'] = adata.obs[['x0','y0']].values
# add on smoothed vectors
movement = smoothed_vectors.set_index('segment')[['unit_u','unit_v']]
movement.index = movement.index.astype(str)
adata.obs[movement.columns] = movement

# compute the angles
smoothed_vectors['angle'] = np.arctan(smoothed_vectors['unit_u'] / smoothed_vectors['unit_v'])
smoothed_vectors.loc[smoothed_vectors['unit_u'] < 0, 'angle'] += np.pi
smoothed_vectors['angle'] = (smoothed_vectors['angle'] + 2 * np.pi) % (2 * np.pi)
angles = smoothed_vectors.set_index('segment')['angle']
angles.index = angles.index.astype(str)
adata.obs['angle'] = angles
# compute all distances
dist = cdist(adata.obsm['spatial'], adata.obsm['spatial']) * 0.390625
dist = pd.DataFrame(dist, index=adata.obs.index, columns=adata.obs.index)

In [ ]:
# define a method to track the streams
def iter_angle(fig, src_x, src_y, src_angle):
    # compute distance
    dist_to_idx = cdist(adata.obsm['spatial'], [[src_x, src_y]])[:, 0]
    close_idxs = adata.obs.index[dist_to_idx < max_radius]
    close_idxs = close_idxs[close_idxs != idx]
    close_angles = adata.obs.loc[close_idxs, 'angle'].dropna()
    
    # angles may be computed in differences from clock or counter clock wise
    # only keep indices within 45 degrees
    angle_diffs1 = (close_angles - src_angle).abs()
    angle_diffs2 = (close_angles - (src_angle + 2 * np.pi)).abs()
    angle_diffs = pd.concat([angle_diffs2, angle_diffs1], axis=1).min(axis=1)
    valid_idxs = angle_diffs.index[angle_diffs < view_angle]
    
    # then the cells themselves must be within 45 degrees of the original cell
    xdiff = adata[valid_idxs].obsm['spatial'][:, 0] - src_x
    ydiff = adata[valid_idxs].obsm['spatial'][:, 1] - src_y
    angle_to_src = pd.Series((np.arctan(ydiff / xdiff) + 2 * np.pi) % (2 * np.pi), index=valid_idxs)
    angle_to_src[xdiff < 0] += np.pi
    angle_diffs1 = (angle_to_src - src_angle).abs()
    angle_diffs2 = (angle_to_src - (src_angle + 2 * np.pi)).abs()
    angle_diffs_to_src = pd.concat([angle_diffs2, angle_diffs1], axis=1).min(axis=1)
    next_idxs = valid_idxs[angle_diffs_to_src < view_angle]
    if len(next_idxs) == 0: return
    
    # derive weights for weighted position and direction
    x, y = adata[next_idxs].obsm['spatial'].mean(0)
    angle = close_angles.loc[next_idxs].mean(0)
    
    # plot data to visualize this specific stream (this is optional and can be deleted,
    # we delete this part when we are computing streams in mass)
    ax = fig.gca()
    ax.scatter(src_x, src_y, color='dodgerblue', edgecolor='k', marker='s', zorder=2, s=25)
    ax.scatter(adata[next_idxs].obsm['spatial'][:, 0], adata[next_idxs].obsm['spatial'][:, 1],
               vmin=0, vmax=np.pi, cmap='Blues_r', zorder=1, c=angle_diffs_to_src.loc[next_idxs], s=25)
    for idx_ in next_idxs:
        x1, y1 = adata[idx_].obsm['spatial'][:, 0][0], adata[idx_].obsm['spatial'][:, 1][0]
        ax.plot([src_x, x1], [src_y, y1], color='lightgray', linestyle='--', lw=0.5, zorder=0, alpha=0.5)
    
    return fig, x, y, angle, next_idxs

In [ ]:
# compute all streams within this biopsy
max_radius = 90
view_angle = np.pi / 4
streams = {}
for idx in tqdm(adata.obs.index):
    # intially gather data
    src_x = adata[idx].obsm['spatial'][:, 0][0]
    src_y = adata[idx].obsm['spatial'][:, 1][0]
    src_angle = adata.obs.loc[idx, 'angle']
    next_idxs_list = []
    for _ in range(25):
        try:
            prev_x, prev_y = src_x, src_y
            fig, src_x, src_y, src_angle, next_idxs = iter_angle(fig, src_x, src_y, src_angle)
            next_idxs_list.append(next_idxs)
        except:
            break
    if len(next_idxs_list) < 5:
        continue
    streams[idx] = next_idxs_list
stream_pos = {k: [adata[el].obsm['spatial'].mean(0) for el in els] for k, els in streams.items()}

In [ ]:
# plot the streams onto the cell type colored spatial plot
fig, ax = plt.subplots(figsize=[6, 6])
_ = ax.grid(False)
ax = sc.pl.spatial(adata, spot_size=15, show=False, ax=ax, alpha=0.5)[0]
ax = sc.pl.spatial(adata[list(streams.keys())], spot_size=20, ax=ax, color=['cell_type'],
                   show=False, alpha=1, edgecolor='k', linewidth=0.25)[0]
for v in stream_pos.values():
    v = np.array(v)
    ax.plot(v[:, 0], v[:, 1], color='k', alpha=0.25, lw=0.5, linestyle='--')

# highlight the origins
fig, ax = plt.subplots(figsize=[6, 6])
_ = ax.grid(False)
ax = sc.pl.spatial(adata, spot_size=15, show=False, ax=ax, alpha=0.5)[0]
ax = sc.pl.spatial(adata[list(streams.keys())], spot_size=20, ax=ax, color=['cell_type'],
                   show=False, alpha=1, edgecolor='k', linewidth=0.25)[0]
for v in stream_pos.values():
    v = np.array(v)
    ax.plot(v[0, 0], v[0, 1], alpha=0.25, marker='s', markersize=2, color='b')

# highlight the termini
fig, ax = plt.subplots(figsize=[6, 6])
_ = ax.grid(False)
ax = sc.pl.spatial(adata, spot_size=15, show=False, ax=ax, alpha=0.5)[0]
ax = sc.pl.spatial(adata[list(streams.keys())], spot_size=20, ax=ax, color=['cell_type'],
                   show=False, alpha=1, edgecolor='k', linewidth=0.25)[0]
for v in stream_pos.values():
    v = np.array(v)
    ax.plot(v[-1, 0], v[-1, 1], alpha=0.25, marker='o', markersize=2, color='r')

In [ ]:
# assemble expression
timepoints = ['PAST','PRESENT','FUTURE']
exprs, names, pos = [], [], []
for k, next_idxs_list in tqdm(streams.items()):
    if len(next_idxs_list) < 3:
        continue
    n_steps = len(next_idxs_list) - 3 + 1
    for start in range(n_steps):
        # gather data and create a name
        idxs_list = next_idxs_list[start:start+3]
        name = ';'.join([','.join(els) for els in idxs_list])
        # skip if already identified
        if name in names:
            continue
        # retrieve expression data
        expr = []
        for idxs, timepoint in zip(idxs_list, timepoints):
            expr.append(pd.Series(np.log10(adata[idxs].X + 1).mean(0), index=adata.var.index + ':' + timepoint))
        expr = pd.concat(expr, axis=0)
        expr.name = name
        exprs.append(expr)
        names.append(name)
        # get the spatial position of the center
        pos.append(adata[idxs_list[1]].obsm['spatial'].mean(0))
expr = pd.concat(exprs, axis=1).T

In [ ]:
# concatenate all of the expression profiles
expr = pd.concat(exprs, axis=1).T
X = cdist(expr.iloc[:, :-expr.shape[1] // 3].values, expr.iloc[:, expr.shape[1] // 3:].values)
# compute kNN graph and PHATE projection via past-present - present-future distances
knn = kneighbors_graph(X, n_neighbors=15+1, include_self=True, metric='euclidean', mode='connectivity')
phate_operator = phate.PHATE(n_jobs=20, random_state=0, knn_dist='precomputed_affinity', n_landmark=2000)
stream_phate = phate_operator.fit_transform(knn)
# instantiate anndata object based on the trajectory expression profiles
edata = ad.AnnData(expr)
edata.obsm['X_phate'] = stream_phate

In [ ]:
import igraph as ig
import leidenalg as la
# perform leiden clustering on kNN graph (symmetrized)
def sparse_to_igraph(sparse_mx):
    sparse_mx = sparse_mx.tocoo()
    edges = list(zip(sparse_mx.row, sparse_mx.col))
    weights = sparse_mx.data
    g = ig.Graph(n=sparse_mx.shape[0], edges=edges, directed=False)
    g.es['weight'] = weights
    return g
knn_sym = 0.5 * (knn + knn.T)
g = sparse_to_igraph(knn_sym)
partition = la.find_partition(g, la.RBConfigurationVertexPartition, weights=g.es['weight'],
                              seed=0, resolution_parameter=0.8)
edata.obs['leiden'] = pd.Series(partition.membership, index=edata.obs.index).astype(str)
sc.pl.embedding(edata, basis='X_phate', color=['leiden'])

In [ ]:
# derive thelist of paths from the streams
paths = []
for k, next_idxs_list in tqdm(streams.items()):
    # skip if there are not enough steps
    if len(next_idxs_list) < 3:
        continue
    n_steps = len(next_idxs_list) - 3 + 1
    names = []
    for start in range(n_steps):
        # gather data and create a name
        idxs_list = next_idxs_list[start:start+3]
        name = ';'.join([','.join(els) for els in idxs_list])
        names.append(name)
    paths.append(names)
    
# retrieve the origin and termini from the paths
origins_idxs, termini_idxs = [], []
for path in tqdm(paths):
    origins_idxs.append(path[0])
    termini_idxs.append(path[-1])

In [ ]:
# let's pick a specific leiden to work through this exercise
mask_o = edata.obs.index.intersection(origins_idxs)
mask_t = edata.obs.index.intersection(termini_idxs)

# compute distances between the points
expr = pd.DataFrame(edata.X, index=edata.obs.index, columns=edata.var.index)
dist_to_o = cdist(expr.loc[mask_o].iloc[:, :-expr.shape[1] // 3].values, expr.iloc[:, expr.shape[1] // 3:].values)
dist_to_t = cdist(expr.loc[mask_t].iloc[:, :-expr.shape[1] // 3].values, expr.iloc[:, expr.shape[1] // 3:].values)
avg_dist_to_o = dist_to_o.mean(0)
avg_dist_to_t = dist_to_t.mean(0)
dist_o_to_t = avg_dist_to_o - avg_dist_to_t

In [ ]:
# identify the genes that correlate overall
expr = expr[expr.columns[expr.columns.str.endswith(':PRESENT')]].copy()
expr.columns = expr.columns.str.replace(':PRESENT','')
stats = pd.DataFrame(columns=['r','p'])
for el in expr.columns:
    r, p = ss.pearsonr(dist_o_to_t, expr[el])
    stats.loc[el] = r, p

In [ ]:
# look at the correlation with expression
markers = ['HLA DPDQDR','CD4','CD45','HLA class 1 A, B, and C, Na-K-ATPase','CD8','CD14']
fig, ax = plt.subplots(figsize=[4, 4])
_ = ax.grid(False)
colors = sns.color_palette('Set2', len(markers)).as_hex()
for marker, color in zip(markers, colors):
    xs, ys = dist_o_to_t, expr[marker]
    xl, yl, yci = [], [], []
    step_size = 10; max_val = round(100 / step_size)
    for step in range(1, max_val+1):
        vmin = np.percentile(xs, (step-1)*step_size)
        if step == 1: vmin = xs.min()
        vmax = np.percentile(xs, step*step_size)
        if step == max_val: vmax = xs.max() + 1
        mask = (xs >= vmin) & (xs < vmax)
        xl.append(step*5)
        yl.append(ys[mask].mean())
        yci.append(ys[mask].std()*1.96/np.sqrt(sum(mask)))
    yl = np.array(yl)
    yl = yl - yl.min()
    yl = yl / yl.max()
    # plot the values
    marker = marker.replace('HLA class 1 A, B, and C, Na-K-ATPase','HLA ABC, NaKATPase')
    _ = ax.plot(xl, yl, color=color, lw=2, zorder=1, linestyle='-', label=marker.split(':')[0])
    _ = ax.fill_between(xl, yl+yci, yl-yci, color=color, zorder=0, lw=1, linestyle='--', alpha=0.5)
_ = ax.set(xlabel='Origin $\leftarrow \\rightarrow$ Termini', ylabel='Protein Expression')
_ = ax.legend(bbox_to_anchor=(1, .5), bbox_transform=ax.transAxes, loc='center left', frameon=False)

In [ ]:
# grab the relevant data and determine the order
mask_o = edata.obs.index.isin(mask_o)
mask_t = edata.obs.index.isin(mask_t)
stats_list = []
for leiden in edata.obs['leiden'].unique():
    # compute distances between the points for this specific leiden
    mask_clust = edata.obs['leiden'] == leiden
    expr = pd.DataFrame(edata[mask_clust].X, index=edata.obs.index[mask_clust], columns=edata.var.index)
    dist_to_o = cdist(expr.loc[mask_o&mask_clust].iloc[:, :-expr.shape[1] // 3].values, expr.loc[mask_clust].iloc[:, expr.shape[1] // 3:].values)
    dist_to_t = cdist(expr.loc[mask_t&mask_clust].iloc[:, :-expr.shape[1] // 3].values, expr.loc[mask_clust].iloc[:, expr.shape[1] // 3:].values)
    avg_dist_to_o = dist_to_o.mean(0)
    avg_dist_to_t = dist_to_t.mean(0)
    dist_o_to_t = avg_dist_to_o - avg_dist_to_t
    
    # identify the genes that correlate overall
    expr = expr[expr.columns[expr.columns.str.endswith(':PRESENT')]].copy()
    expr.columns = expr.columns.str.replace(':PRESENT','')
    stats = pd.DataFrame(columns=['r','p','leiden'])
    for el in expr.columns:
        r, p = ss.pearsonr(dist_o_to_t, expr[el])
        stats.loc[el] = r, p, leiden
    stats_list.append(stats)
stats = pd.concat(stats_list, axis=0)

# aggregate pearson correlations and p-values
data_r = stats.reset_index().pivot_table(index='index', columns='leiden', values='r').T
data_p = stats.reset_index().pivot_table(index='index', columns='leiden', values='p').T
data_r_nan = data_r.fillna(0)
data_p_nan = data_p.fillna(1)

# plot the aggregate correlations
mask = (data_p_nan >= 0.05) | (data_r.isna())
g = sns.clustermap(data_r_nan, method='ward', figsize=[8, 9], dendrogram_ratio=(.1, .15), mask=mask,
                   cbar_pos=(-0.07, .89, .01, .07), vmin=-0.8, vmax=0.8, cmap='coolwarm', xticklabels=1, yticklabels=1)
_ = g.ax_heatmap.grid(False); _ = g.ax_cbar.grid(False); _ = g.ax_heatmap.set_facecolor('lightgray')
_ = g.ax_heatmap.tick_params(axis='y', labelrotation=0)

In [ ]:
# compute distances between the points for this specific leiden
mask_clust = edata.obs['leiden'] == '7'
expr = pd.DataFrame(edata[mask_clust].X, index=edata.obs.index[mask_clust], columns=edata.var.index)
dist_to_o = cdist(expr.loc[mask_o&mask_clust].iloc[:, :-expr.shape[1] // 3].values, expr.loc[mask_clust].iloc[:, expr.shape[1] // 3:].values)
dist_to_t = cdist(expr.loc[mask_t&mask_clust].iloc[:, :-expr.shape[1] // 3].values, expr.loc[mask_clust].iloc[:, expr.shape[1] // 3:].values)
avg_dist_to_o = dist_to_o.mean(0)
avg_dist_to_t = dist_to_t.mean(0)
dist_o_to_t = avg_dist_to_o - avg_dist_to_t
expr = expr[expr.columns[expr.columns.str.endswith(':PRESENT')]].copy()
expr.columns = expr.columns.str.replace(':PRESENT','')

# look at the correlation with expression
markers = ['Helios','FOXP3','CD45RO','Ki67']
fig, ax = plt.subplots(figsize=[4, 4])
_ = ax.grid(False)
colors = sns.color_palette('Set1', len(markers)).as_hex()[:4]
for marker, color in zip(markers, colors):
    xs, ys = dist_o_to_t, expr[marker]
    xl, yl, yci = [], [], []
    step_size = 10; max_val = round(100 / step_size)
    for step in range(1, max_val+1):
        vmin = np.percentile(xs, (step-1)*step_size)
        if step == 1: vmin = xs.min()
        vmax = np.percentile(xs, step*step_size)
        if step == max_val: vmax = xs.max() + 1
        mask = (xs >= vmin) & (xs < vmax)
        xl.append(step*5)
        yl.append(ys[mask].mean())
        yci.append(ys[mask].std()*1.96/np.sqrt(sum(mask)))
    yl = np.array(yl)
    yl = yl - yl.min()
    yl = yl / yl.max()
    # plot the values
    _ = ax.plot(xl, yl, color=color, lw=2, zorder=1, linestyle='-', label=marker.split(':')[0])
    _ = ax.fill_between(xl, yl+yci, yl-yci, color=color, zorder=0, lw=1, linestyle='--', alpha=0.5)
_ = ax.set(xlabel='Origin $\leftarrow \\rightarrow$ Termini', ylabel='Protein Expression')
_ = ax.legend(bbox_to_anchor=(1, .5), bbox_transform=ax.transAxes, loc='center left', frameon=False)

In [ ]:
# compute distances between the points for this specific leiden
mask_clust = edata.obs['leiden'] == '8'
expr = pd.DataFrame(edata[mask_clust].X, index=edata.obs.index[mask_clust], columns=edata.var.index)
dist_to_o = cdist(expr.loc[mask_o&mask_clust].iloc[:, :-expr.shape[1] // 3].values, expr.loc[mask_clust].iloc[:, expr.shape[1] // 3:].values)
dist_to_t = cdist(expr.loc[mask_t&mask_clust].iloc[:, :-expr.shape[1] // 3].values, expr.loc[mask_clust].iloc[:, expr.shape[1] // 3:].values)
avg_dist_to_o = dist_to_o.mean(0)
avg_dist_to_t = dist_to_t.mean(0)
dist_o_to_t = avg_dist_to_o - avg_dist_to_t
expr = expr[expr.columns[expr.columns.str.endswith(':PRESENT')]].copy()
expr.columns = expr.columns.str.replace(':PRESENT','')

# look at the correlation with expression
markers = ['CD14','CD4','CD11b','CD68']
fig, ax = plt.subplots(figsize=[4, 4])
_ = ax.grid(False)
colors = np.array(sns.color_palette('Set1', 9).as_hex())[[4, 6, 7, 8]]
for marker, color in zip(markers, colors):
    xs, ys = dist_o_to_t, expr[marker]
    xl, yl, yci = [], [], []
    step_size = 10; max_val = round(100 / step_size)
    for step in range(1, max_val+1):
        vmin = np.percentile(xs, (step-1)*step_size)
        if step == 1: vmin = xs.min()
        vmax = np.percentile(xs, step*step_size)
        if step == max_val: vmax = xs.max() + 1
        mask = (xs >= vmin) & (xs < vmax)
        xl.append(step*5)
        yl.append(ys[mask].mean())
        yci.append(ys[mask].std()*1.96/np.sqrt(sum(mask)))
    yl = np.array(yl)
    yl = yl - yl.min()
    yl = yl / yl.max()
    # plot the values
    _ = ax.plot(xl, yl, color=color, lw=2, zorder=1, linestyle='-', label=marker.split(':')[0])
    _ = ax.fill_between(xl, yl+yci, yl-yci, color=color, zorder=0, lw=1, linestyle='--', alpha=0.5)
_ = ax.set(xlabel='Origin $\leftarrow \\rightarrow$ Termini', ylabel='Protein Expression')
_ = ax.legend(bbox_to_anchor=(1, .5), bbox_transform=ax.transAxes, loc='center left', frameon=False)

### Fig. 6 and S7 ("Streams" in a Biopsy of Lung Cancer)

In [ ]:
from collections import defaultdict
from tqdm import tqdm
# Read in cell boundary data
df = pd.read_parquet('/PATH/TO/XENIUM_LUNG_CELL_BOUNARDIES_PARQUET')
# Map cell id to segmentation id
cell_ids = df['cell_id'].unique()
c2i = {x: i for i, x in enumerate(cell_ids)}
df['cell_id'] = df['cell_id'].map(c2i)
df = df[['vertex_x','vertex_y','cell_id']].values
# Convert micron to pixels
df[:, :2] /= 0.2125
# Instantiate tracking dictionary
segmentation_dict = defaultdict(list)
# Process all cell ids
prev_cell_id = None
for idx in tqdm(range(df.shape[0])):
    x, y, cell_id = df[idx, :]
    segmentation_dict[int(cell_id)].append([x, y])

In [ ]:
# Path to the pyramid OME-TIFF
marker_file = '/PATH/TO/MORPHOLOGY_OME_0003_TIFF'
# Derive list of marker genes from the XML
print(f'Reading in TIF files from {marker_file}')
with tifffile.TiffFile(marker_file) as tif:
    ome_metadata = tif.ome_metadata
root = ET.fromstring(ome_metadata)
namespace = {'ome': 'http://www.openmicroscopy.org/Schemas/OME/2016-06'}
# Find all channel names
channel_names = []
for image in root.findall('ome:Image', namespace):
    for channel in image.find('ome:Pixels', namespace).findall('ome:Channel', namespace):
        name = channel.get('Name')
        channel_names.append(name)
# Retrieve full image
print(f'Loading OME-TIFF stack from: {marker_file}')
with tifffile.TiffFile(marker_file) as tif:
    marker_imstack = tif.asarray()
    num_layers, img_height, img_width = marker_imstack.shape
    print(f'Loaded image stack with {num_layers} layers, height: {img_height}, width: {img_width}')

In [ ]:
# Retrieve intensities (Y, X)
intensities = marker_imstack[channel_names.index('alphaSMA/Vimentin'), :, :]
del marker_imstack
# Set accelerator device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
intensities_acc = torch.tensor(intensities, device=device, dtype=torch.float32)
# Generate vector
vectors = generate_vector_field(segmentation_dict, intensities_acc, device)
vectors_stor = vectors.copy()
# Subset for a small part of the tumor
mask = (vectors['x0'] >= 10000) & (vectors['x0'] < 20000)
mask = mask & (vectors['y0'] >= 7000) & (vectors['y0'] < 13000)
vectors = vectors.loc[mask].reset_index().iloc[:, 1:]

In [ ]:
# define chunk parameters
chunk_size = 2500
start, end = 0, chunk_size
# instantiate new columns in vectors
keys = ['smooth_u','smooth_v','smooth_magnitude','unit_u','unit_v']
vectors[keys] = np.nan
# process each chunk
while start < vectors.shape[0]:
    # subset for the relevant data
    mask = (vectors.index >= start) & (vectors.index < end)
    df = vectors.loc[mask]
    # calculate distance matrix between each object
    dist_segment = cdist(df[['x0','y0']], df[['x0','y0']])
    dist_segment = pd.DataFrame(dist_segment, index=df.index, columns=df.index)
    # only consider vectors above a minimum magnitude
    mask = df['magnitude'] >= MIN_VECTOR_MAGNITUDE
    df = df.loc[mask]
    # smooth vectors by spatial distance
    df['smooth_u'] = spatial_smooth(df['xv'], SMOOTH_ITERS, dist_segment, SMOOTH_DIST_LIMIT_MICRONS / 0.2125)
    df['smooth_v'] = spatial_smooth(df['yv'], SMOOTH_ITERS, dist_segment, SMOOTH_DIST_LIMIT_MICRONS / 0.2125)
    # normalize into unit vectors
    df['smooth_magnitude'] = np.linalg.norm(df[['smooth_u', 'smooth_v']], axis=1)
    df['unit_u'] = df['smooth_u'] / df['smooth_magnitude']
    df['unit_v'] = df['smooth_v'] / df['smooth_magnitude']
    # add to the object
    vectors.loc[df.index, keys] = df[keys]
    # proceed to next chunk
    start += chunk_size
    end += chunk_size
vectors_save = vectors.copy()

In [ ]:
# read in the data
adata = sc.read_10x_h5('/PATH/TO/CELL_FEATURE_MATRIX_H5')
adata.obs['seg_id'] = adata.obs.index.map(c2i)
# the original vectors are then re-attached to the anndata object
vectors = vectors_stor.copy()
mask = (vectors['x0'] >= 10000) & (vectors['x0'] < 20000)
mask = mask & (vectors['y0'] >= 7000) & (vectors['y0'] < 13000)
adata.obs[vectors.columns] = vectors.loc[adata.obs['seg_id']].values
# add in spatial coordinates
adata.obsm['spatial'] = adata.obs[['x0','y0']].values
# save raw counts
adata.layers['raw_counts'] = adata.X.copy()
# normalize data
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
# subset for the ROI
adata = adata[mask].copy()
vectors = vectors_save.copy()
adata.obs.index = vectors.index
sc.pp.filter_cells(adata, min_counts=1)

# compute neighborhoods
sc.tl.pca(adata)
sc.pp.neighbors(adata, random_state=0)
sc.tl.umap(adata, random_state=0)
sc.tl.leiden(adata, flavor='igraph', n_iterations=2, random_state=0)
sc.pl.umap(adata, color=['leiden'])
sc.pl.umap(adata, color=['EPCAM','ACE2','VWF','PECAM1','MRC1','CD68','TRAC','CD3D','MZB1','CD79A','COL5A2','ACTA2'], use_raw=False, ncols=6)

# plot the DEGs per cell type
genes = ['EPCAM','ACE2','VWF','PECAM1','MRC1','CD68','TRAC','CD3D','MZB1','CD79A','COL5A2','ACTA2']
sc.pl.dotplot(adata, groupby='leiden', var_names=genes, use_raw=False, cmap='Blues', dot_max=0.5, vmax=3,
              categories_order=['6','7','5','4','8','0','9','1','10','3','2'])

# project angles onto the data
fig, ax = plt.subplots(figsize=[8, 8])
ax = sc.pl.spatial(adata, spot_size=0, show=False, ax=ax)
ax = sc.pl.spatial(adata, spot_size=50, ax=ax[0], color=['leiden'], show=False, alpha=0.8)[0]
ax.quiver(adata.obs['x0'], adata.obs['y0'], adata.obs['unit_u'], adata.obs['unit_v'],
          width=QUIVER_WIDTH * 0.4, scale=QUIVER_SCALE * 3, alpha=0.8)

In [ ]:
# compute the angles
smoothed_vectors['angle'] = np.arctan(smoothed_vectors['unit_u'] / smoothed_vectors['unit_v'])
smoothed_vectors.loc[smoothed_vectors['unit_u'] < 0, 'angle'] += np.pi
smoothed_vectors['angle'] = (smoothed_vectors['angle'] + 2 * np.pi) % (2 * np.pi)
angles = smoothed_vectors['angle']
angles.index = angles.index.astype(str)
adata.obs['angle'] = angles.loc[adata.obs.index].values
# compute all distances
dist = cdist(adata.obsm['spatial'], adata.obsm['spatial']) * 0.2125
dist = pd.DataFrame(dist, index=adata.obs.index, columns=adata.obs.index)

# redefine stream identifying function
def iter_angle(fig, src_x, src_y, src_angle):
    # compute distance
    dist_to_idx = cdist(adata.obsm['spatial'], [[src_x, src_y]])[:, 0]
    close_idxs = adata.obs.index[dist_to_idx < max_radius]
    close_idxs = close_idxs[close_idxs != idx]
    close_angles = adata.obs.loc[close_idxs, 'angle'].dropna()
    
    # angles may be computed in differences from clock or counter clock wise
    # only keep indices within 45 degrees
    angle_diffs1 = (close_angles - src_angle).abs()
    angle_diffs2 = (close_angles - (src_angle + 2 * np.pi)).abs()
    angle_diffs = pd.concat([angle_diffs2, angle_diffs1], axis=1).min(axis=1)
    valid_idxs = angle_diffs.index[angle_diffs < view_angle]
    
    # then the cells themselves must be within 45 degrees of the original cell
    xdiff = adata[valid_idxs].obsm['spatial'][:, 0] - src_x
    ydiff = adata[valid_idxs].obsm['spatial'][:, 1] - src_y
    angle_to_src = pd.Series((np.arctan(ydiff / xdiff) + 2 * np.pi) % (2 * np.pi), index=valid_idxs)
    angle_to_src[xdiff < 0] += np.pi
    angle_diffs1 = (angle_to_src - src_angle).abs()
    angle_diffs2 = (angle_to_src - (src_angle + 2 * np.pi)).abs()
    angle_diffs_to_src = pd.concat([angle_diffs2, angle_diffs1], axis=1).min(axis=1)
    next_idxs = valid_idxs[angle_diffs_to_src < view_angle]
    if len(next_idxs) == 0: return
    
    # derive weights for weighted position and direction
    x, y = adata[next_idxs].obsm['spatial'].mean(0)
    angle = close_angles.loc[next_idxs].mean(0)
    
    # plot data
    ax = fig.gca()
    ax.scatter(src_x, src_y, color='dodgerblue', edgecolor='k', marker='s', zorder=2, s=25)
    ax.scatter(adata[next_idxs].obsm['spatial'][:, 0], adata[next_idxs].obsm['spatial'][:, 1],
               vmin=0, vmax=np.pi, cmap='Blues_r', zorder=1, c=angle_diffs_to_src.loc[next_idxs], s=25)
    for idx_ in next_idxs:
        x1, y1 = adata[idx_].obsm['spatial'][:, 0][0], adata[idx_].obsm['spatial'][:, 1][0]
        ax.plot([src_x, x1], [src_y, y1], color='lightgray', linestyle='--', lw=0.5, zorder=0, alpha=0.5)
    
    return fig, x, y, angle, next_idxs

In [ ]:
# identify streams in the data
max_radius = 90
view_angle = np.pi / 4
streams = {}
for idx in tqdm(adata.obs.index):
    src_x = adata[idx].obsm['spatial'][:, 0][0]
    src_y = adata[idx].obsm['spatial'][:, 1][0]
    src_angle = adata.obs.loc[idx, 'angle']
    next_idxs_list = []
    for _ in range(25):
        try:
            prev_x, prev_y = src_x, src_y
            fig, src_x, src_y, src_angle, next_idxs = iter_angle(fig, src_x, src_y, src_angle)
            ax.plot([prev_x, src_x], [prev_y, src_y], color='k', linestyle='-', lw=1, zorder=1, alpha=1)
            next_idxs_list.append(next_idxs)
        except:
            break
    if len(next_idxs_list) < 5:
        continue
    streams[idx] = next_idxs_list
stream_pos = {k: [adata[el].obsm['spatial'].mean(0) for el in els] for k, els in streams.items()}

# plot the cells associated with streams and the streams themselves
fig, ax = plt.subplots(figsize=[6, 6])
_ = ax.grid(False)
ax = sc.pl.spatial(adata, spot_size=50, show=False, ax=ax, alpha=0.5)[0]
ax = sc.pl.spatial(adata[list(streams.keys())], spot_size=50, ax=ax, color=['leiden'],
                   show=False, alpha=1, edgecolor='k', linewidth=0.25)[0]
for v in stream_pos.values():
    v = np.array(v)
    ax.plot(v[:, 0], v[:, 1], color='k', alpha=0.25, lw=0.5, linestyle='--')

# which cell types are enriched in streams compared to not
idxs = list(set(streams.keys()))
idxs += list(set([el for vs in streams.values() for v in vs for el in v]))

# compare the distribution of leidens
cats = adata.obs['leiden'].cat.categories.tolist()
mask = adata.obs.index.isin(idxs)
counts_all = adata.obs.loc[~mask, 'leiden'].value_counts().loc[cats]
counts_set = adata.obs.loc[mask, 'leiden'].value_counts().loc[cats]
fig, ax = plt.subplots(); _ = ax.pie(counts_all)
fig, ax = plt.subplots(); _ = ax.pie(counts_set)
# compute p-value
import scipy.stats as ss
p = ss.chi2_contingency(pd.concat([counts_set, counts_all], axis=1)).pvalue
print(p)

# evaluate for any enrichment of T cells
diff = (counts_set / counts_set.sum() * 100) - (counts_all / counts_all.sum() * 100)
# read in the data
fig, ax = plt.subplots()
_ = ax.grid(False)
_ = ax.bar(diff.index, diff, color=adata.uns['leiden_colors'], edgecolor='k', linewidth=1.5)
_ = ax.axhline(0, color='k', linewidth=1.5)
_ = ax.set(xlabel='Leiden', ylabel='Percentage (%) Difference\nNear and Far to Streams')

# look at cells in the stream
adata.obs['in_stream'] = 'No'
adata.obs.loc[idxs, 'in_stream'] = 'Yes'
# look at the DEGs in and out of streams
sc.tl.rank_genes_groups(adata, method='wilcoxon', use_raw=False, groupby='in_stream', groups=['Yes'], reference='No')
genes = pd.DataFrame(adata.uns['rank_genes_groups']['names'])['Yes']
fdrs = pd.DataFrame(adata.uns['rank_genes_groups']['pvals_adj'])['Yes']
stats = pd.concat([genes, fdrs], axis=1)
stats.columns = ['gene','fdr']
stats = stats.set_index('gene')
stats['logfc'] = pd.Series(adata[adata.obs['in_stream'] == 'Yes'].X.mean(0).A1 - \
adata[adata.obs['in_stream'] == 'No'].X.mean(0).A1, index=adata.var.index)
# perform multiple hypothesis correction
import statsmodels.api as sm
stats['fdr'] = pd.Series(sm.stats.fdrcorrection(stats['p'])[1], index=stats.index)

# retrieve color for genes
from matplotlib.cm import get_cmap
from matplotlib.colors import to_hex
genes = ['CXCR4','CCL5','CXCL9','TRAC','CD3E','CD3D','EPCAM','KRT7','EGFR']
heights = stats.loc[genes, 'logfc'].sort_values(); colors = heights.copy()
colors += 0.8; colors /= 1.6; colors[colors < 0] = 0; colors[colors > 1] = 1
cmap = get_cmap('coolwarm'); colors = [to_hex(cmap(el)) for el in colors]
# read in the data
fig, ax = plt.subplots(figsize=[4, 4])
_ = ax.grid(False)
_ = ax.bar(heights.index, heights, edgecolor='k', linewidth=1.5, color=colors)
_ = ax.axhline(0, color='k', linewidth=1.5)
_ = ax.set_xlim(-1, len(genes))
_ = ax.tick_params(axis='x', labelrotation=90)
_ = ax.set(xlabel=None, ylabel='logFC of RNA Expression\nin Stream Near vs. Far Cells')

In [ ]:
# assemble expression
timepoints = ['PAST','PRESENT','FUTURE']
exprs, names, pos = [], [], []
for k, next_idxs_list in tqdm(streams.items()):
    if len(next_idxs_list) < 3:
        continue
    n_steps = len(next_idxs_list) - 3 + 1
    for start in range(n_steps):
        # gather data and create a name
        idxs_list = next_idxs_list[start:start+3]
        name = ';'.join([','.join(els) for els in idxs_list])
        # skip if already identified
        if name in names:
            continue
        # retrieve expression data
        expr = []
        for idxs, timepoint in zip(idxs_list, timepoints):
            expr.append(pd.Series(adata[idxs].X.mean(0).A1, index=adata.var.index + ':' + timepoint))
        expr = pd.concat(expr, axis=0)
        expr.name = name
        exprs.append(expr)
        names.append(name)
        # get the spatial position of the center
        pos.append(adata[idxs_list[1]].obsm['spatial'].mean(0))
expr = pd.concat(exprs, axis=1).T

# concatenate all of the expression profiles
expr = pd.concat(exprs, axis=1).T
X = cdist(expr.iloc[:, :-expr.shape[1] // 3].values, expr.iloc[:, expr.shape[1] // 3:].values)
# compute kNN graph and PHATE projection via past-present - present-future distances
knn = kneighbors_graph(X, n_neighbors=15+1, include_self=True, metric='euclidean', mode='connectivity')
phate_operator = phate.PHATE(n_jobs=20, random_state=0, knn_dist='precomputed_affinity', n_landmark=2000)
stream_phate = phate_operator.fit_transform(knn)
# instantiate anndata object based on the trajectory expression profiles
edata = ad.AnnData(expr)
edata.obsm['X_phate'] = stream_phate

# perform leiden clustering on kNN graph (symmetrized)
knn_sym = 0.5 * (knn + knn.T)
g = sparse_to_igraph(knn_sym)
partition = la.find_partition(g, la.RBConfigurationVertexPartition, weights=g.es['weight'],
                              seed=0, resolution_parameter=0.8)
edata.obs['leiden'] = pd.Series(partition.membership, index=edata.obs.index).astype(str)
sc.pl.embedding(edata, basis='X_phate', color=['leiden'])

In [ ]:
# derive thelist of paths from the streams
paths = []
for k, next_idxs_list in tqdm(streams.items()):
    # skip if there are not enough steps
    if len(next_idxs_list) < 3:
        continue
    n_steps = len(next_idxs_list) - 3 + 1
    names = []
    for start in range(n_steps):
        # gather data and create a name
        idxs_list = next_idxs_list[start:start+3]
        name = ';'.join([','.join(els) for els in idxs_list])
        names.append(name)
    paths.append(names)
    
# retrieve the origin and termini from the paths
origins_idxs, termini_idxs = [], []
for path in tqdm(paths):
    origins_idxs.append(path[0])
    termini_idxs.append(path[-1])

# let's pick a specific leiden to work through this exercise
mask_o = edata.obs.index.intersection(origins_idxs)
mask_t = edata.obs.index.intersection(termini_idxs)

# compute distances between the points
expr = pd.DataFrame(edata.X, index=edata.obs.index, columns=edata.var.index)
dist_to_o = cdist(expr.loc[mask_o].iloc[:, :-expr.shape[1] // 3].values, expr.iloc[:, expr.shape[1] // 3:].values)
dist_to_t = cdist(expr.loc[mask_t].iloc[:, :-expr.shape[1] // 3].values, expr.iloc[:, expr.shape[1] // 3:].values)
avg_dist_to_o = dist_to_o.mean(0)
avg_dist_to_t = dist_to_t.mean(0)
dist_o_to_t = avg_dist_to_o - avg_dist_to_t
edata.obs['density'] = dist_o_to_t
sc.pl.embedding(edata, basis='X_phate', color='density', cmap='RdBu_r', title='Density Origin $\\rightarrow$ Termini', vmin=-2, vmax=2)

# identify the genes that correlate overall
expr = expr[expr.columns[expr.columns.str.endswith(':PRESENT')]].copy()
expr.columns = expr.columns.str.replace(':PRESENT','')
stats = pd.DataFrame(columns=['r','p'])
for el in expr.columns:
    r, p = ss.pearsonr(dist_o_to_t, expr[el])
    stats.loc[el] = r, p

import seaborn as sns
# look at the correlation with expression
markers = ['CXCR4','CD3E','CCL5','PECAM1','MRC1','MZB1']
fig, ax = plt.subplots(figsize=[4, 4])
_ = ax.grid(False)
colors = sns.color_palette('Set2', len(markers)).as_hex()
for marker, color in zip(markers, colors):
    xs, ys = dist_o_to_t, expr[marker]
    xl, yl, yci = [], [], []
    step_size = 10; max_val = round(100 / step_size)
    for step in range(1, max_val+1):
        vmin = np.percentile(xs, (step-1)*step_size)
        if step == 1: vmin = xs.min()
        vmax = np.percentile(xs, step*step_size)
        if step == max_val: vmax = xs.max() + 1
        mask = (xs >= vmin) & (xs < vmax)
        xl.append(step*5)
        yl.append(ys[mask].mean())
        yci.append(ys[mask].std()*1.96/np.sqrt(sum(mask)))
    yl = np.array(yl)
    yl = yl - yl.min()
    yl = yl / yl.max()
    # plot the values
    marker = marker.replace('HLA class 1 A, B, and C, Na-K-ATPase','HLA ABC, NaKATPase')
    _ = ax.plot(xl, yl, color=color, lw=2, zorder=1, linestyle='-', label=marker.split(':')[0])
    _ = ax.fill_between(xl, yl+yci, yl-yci, color=color, zorder=0, lw=1, linestyle='--', alpha=0.5)
_ = ax.set(xlabel='Origin $\leftarrow \\rightarrow$ Termini', ylabel='Protein Expression')
_ = ax.legend(bbox_to_anchor=(1, .5), bbox_transform=ax.transAxes, loc='center left', frameon=False)

# look at the correlation with expression
markers = ['EPCAM','EGFR','KRT7','ACE2','CXCL2','CXCL6']
fig, ax = plt.subplots(figsize=[4, 4])
_ = ax.grid(False)
colors = sns.color_palette('Set1', len(markers)).as_hex()
for marker, color in zip(markers, colors):
    xs, ys = dist_o_to_t, expr[marker]
    xl, yl, yci = [], [], []
    step_size = 10; max_val = round(100 / step_size)
    for step in range(1, max_val+1):
        vmin = np.percentile(xs, (step-1)*step_size)
        if step == 1: vmin = xs.min()
        vmax = np.percentile(xs, step*step_size)
        if step == max_val: vmax = xs.max() + 1
        mask = (xs >= vmin) & (xs < vmax)
        xl.append(step*5)
        yl.append(ys[mask].mean())
        yci.append(ys[mask].std()*1.96/np.sqrt(sum(mask)))
    yl = np.array(yl)
    yl = yl - yl.min()
    yl = yl / yl.max()
    # plot the values
    marker = marker.replace('HLA class 1 A, B, and C, Na-K-ATPase','HLA ABC, NaKATPase')
    _ = ax.plot(xl, yl, color=color, lw=2, zorder=1, linestyle='-', label=marker.split(':')[0])
    _ = ax.fill_between(xl, yl+yci, yl-yci, color=color, zorder=0, lw=1, linestyle='--', alpha=0.5)
_ = ax.set(xlabel='Origin $\leftarrow \\rightarrow$ Termini', ylabel='Protein Expression')
_ = ax.legend(bbox_to_anchor=(1, .5), bbox_transform=ax.transAxes, loc='center left', frameon=False)

In [ ]:
# cluster the cells
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.metrics import silhouette_score

# look at the correlation in the temporal data
genes = edata.var.index[edata.var.index.str.contains(':PRESENT')]
data = pd.DataFrame(edata[:, genes].X.toarray(), columns=genes)
data.columns = data.columns.str.replace(':PRESENT','')
data_r = data.corr(method='pearson')
data_p = data.corr(method=lambda xs, ys: ss.pearsonr(xs, ys)[1])
for idx in data_p.index: data_p.loc[idx, idx] = 0

# cluster the cells which allows us to select t=7
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.metrics import silhouette_score
Z = linkage(data_r, method='ward')
scores = pd.Series()
for t in range(2, 20+1):
    clustersTIM = pd.Series(fcluster(Z, t=t, criterion='maxclust'), index=data_r.index)
    scores.loc[t] = silhouette_score(data_r, clustersTIM)

# choose seven clusters
clustersTIM = pd.Series(fcluster(Z, t=7, criterion='maxclust'), index=data_r.index)
# map to colors
clu2col = {i+1: col for i, col in enumerate(sns.color_palette('Set2', 7).as_hex())}
colorsTIM = clustersTIM.map(clu2col)

# look at the correlation
g = sns.clustermap(data_r, xticklabels=0, yticklabels=0, method='ward', vmin=-0.5, vmax=0.5, cmap='coolwarm',
                   figsize=[8, 8], cbar_pos=(0.03, .87, .01, .08), mask=data_p >= 0.05,
                   row_colors=colorsTIM, col_colors=colorsTIM, colors_ratio=.02)
_ = g.ax_heatmap.grid(False); _ = g.ax_cbar.grid(False)
_ = g.ax_heatmap.set_facecolor('lightgray')
_ = g.ax_row_colors.grid(False); _ = g.ax_col_colors.grid(False)

# work through each of the clusters from RNA
cs = sorted(clustersTIM.unique())
for c in cs:
    genes = clustersTIM.index[clustersTIM == c]
    sc.tl.score_genes(adata, score_name=f'score{c}', gene_list=genes, random_state=0, use_raw=False)
    adata.obs[f'score{c}'] -= adata.obs[f'score{c}'].mean()
    adata.obs[f'score{c}'] /= adata.obs[f'score{c}'].std()
    sc.pl.umap(adata, color=[f'score{c}'], vmin=-2, vmax=2, cmap='coolwarm', title=f'Stream Cluster {c}')

# compute the average expression of each module per trajectory step
density = pd.Series(dist_o_to_t, index=edata.obs.index)
col_scores = adata.obs.columns[adata.obs.columns.str.startswith('score')]
data = []
for obs_idx in tqdm(edata.obs.index):
    data_ = adata.obs.loc[pd.Index(obs_idx.split(';')[1].split(',')).astype(int).astype(str), col_scores].mean(0)
    data_.loc['density'] = density[obs_idx]
    data_.name = obs_idx
    data.append(data_)
data = pd.concat(data, axis=1).T
data -= data.mean(); data /= data.std()

# aggregate per percentile and plot the module scores across trajectory steps
data['perc'] = (data['density'].rank(pct=True)).round(1)
avgs = data.groupby('perc').mean()
ci95s = data.groupby('perc').std().T / np.sqrt(data['perc'].value_counts()) * 1.96
fig, ax = plt.subplots(figsize=[8, 4])
_ = ax.grid(False)
for score, color in zip(data.columns[:-1], sns.color_palette('Set2', 7).as_hex()):
    sns.regplot(x=data['density'], y=data[score], scatter=False, order=4, color=color, label=score[-1])